# Manuscript figures


In [ ]:
from __future__ import annotations

import os
import json
import re
import matplotlib.ticker as ticker
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from adjustText import adjust_text
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

PROJECT_ROOT = Path(os.environ.get(
    "CHD_OUTPUT_ROOT",
    "../outputs",
))
RUN_ROOT = Path(os.environ.get(
    "CHD_RUN_ROOT",
    str(PROJECT_ROOT / "runs" / os.environ.get("CHD_PRIMARY_RUN_ID", "final_rerun_2026")),
))
RESULT_ROOT = RUN_ROOT / "04_results"
OUTPUT_ROOT = PROJECT_ROOT / "plot_only_manual_revision"
FIGURE_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "source_data"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

CNS = {
    "blue": "#3C5488", "red": "#E64B35", "green": "#00A087",
    "teal": "#4DBBD5", "orange": "#F39B7F", "purple": "#8491B4",
    "gray": "#6E6E6E", "light_gray": "#D9D9D9", "black": "#222222",
}
plt.rcParams.update({
    "figure.dpi": 170, "savefig.dpi": 600, "font.family": "DejaVu Sans",
    "font.size": 13.0, "axes.titlesize": 15.0, "axes.labelsize": 13.5,
    "xtick.labelsize": 11.5, "ytick.labelsize": 11.5,
    "legend.fontsize": 11.0, "axes.linewidth": 0.9,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
})

FILE_OVERRIDES = {

}

SENSITIVITY_ROOT = PROJECT_ROOT / "runs" / "sensitivity_coverage90"
SEARCH_ROOTS = []
for candidate in [
    Path(os.environ["CHD_RESULT_ROOT"]) if os.environ.get("CHD_RESULT_ROOT") else None,
    RUN_ROOT / "04_results",
    RUN_ROOT,
]:
    if candidate is not None and candidate.exists() and candidate not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(candidate)

def _locate(filename: str, sensitivity_only: bool = False) -> Path:
    if not sensitivity_only and filename in FILE_OVERRIDES:
        selected = Path(FILE_OVERRIDES[filename])
        if not selected.exists():
            raise FileNotFoundError(selected)
        return selected

    roots = [SENSITIVITY_ROOT] if sensitivity_only else SEARCH_ROOTS
    matches = []
    for root in roots:
        if not root.exists():
            continue
        matches.extend(root.rglob(filename))
    unique = []
    seen = set()
    for path in matches:
        key = str(path.resolve())
        if key in seen or str(path).startswith(str(OUTPUT_ROOT)):
            continue
        if not sensitivity_only and "sensitivity_coverage90" in str(path):
            continue
        seen.add(key)
        unique.append(path)
    if not unique:
        scope = SENSITIVITY_ROOT if sensitivity_only else SEARCH_ROOTS
        raise FileNotFoundError(f"{filename} was not found in {scope}")

    def score(path):
        text = str(path)
        priority = 0
        if str(RUN_ROOT) in text:
            priority += 100
        if "source_data" in path.parts:
            priority += 20
        if "figure_data" in path.parts:
            priority += 15
        if "tables" in path.parts:
            priority += 10
        return priority, path.stat().st_mtime

    unique.sort(key=score, reverse=True)
    if len(unique) > 1:
        print(f"WARNING: Found {len(unique)} matches for {filename}. Using {unique[0]}")
    return unique[0]

def find_unique(filename: str) -> Path:
    return _locate(filename, sensitivity_only=False)

def find_sensitivity_unique(filename: str) -> Path:
    return _locate(filename, sensitivity_only=True)

def require_columns(frame: pd.DataFrame, columns, source: Path) -> None:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{source} Missing columns: {missing}; available columns={list(frame.columns)}")

def clean_label(value: object) -> str:
    text = str(value).replace("_", " ")
    return text.replace("Clinical plus", "Clinical +")

def clean_axis(ax, grid_axis="x") -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color=CNS["light_gray"], linewidth=0.7, alpha=0.75)
    ax.set_axisbelow(True)

def save_show(fig: plt.Figure, stem: str) -> None:
    for extension in ("png", "pdf", "svg"):
        fig.savefig(FIGURE_DIR / f"{stem}.{extension}", dpi=600, bbox_inches="tight")
    print("Saved:", FIGURE_DIR / f"{stem}.png")
    plt.show()
    plt.close(fig)

RECLASS_METRICS = {
    "continuous_nri": ("NRI_Continuous", "NRI_Continuous_Lower95", "NRI_Continuous_Upper95", "Continuous NRI", CNS["green"]),
    "categorical_nri": ("NRI_Categorical", "NRI_Categorical_Lower95", "NRI_Categorical_Upper95", "Categorical NRI", CNS["blue"]),
    "idi": ("IDI", "IDI_Lower95", "IDI_Upper95", "IDI", CNS["orange"]),
}

def plot_reclassification_one(*, filename, horizon, metric, label_column, stem, filters=None):
    source = find_unique(filename)
    data = pd.read_csv(source)
    filters = filters or {}
    for column, value in filters.items():
        require_columns(data, [column], source)
        data = data.loc[data[column].astype(str).eq(str(value))]
    require_columns(data, ["Horizon_Years", label_column], source)
    data = data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"], errors="coerce"), float(horizon))].copy()
    estimate, lower, upper, metric_label, color = RECLASS_METRICS[metric]
    require_columns(data, [estimate, lower, upper], source)
    for column in (estimate, lower, upper):
        data[column] = pd.to_numeric(data[column], errors="coerce")
    data = data.dropna(subset=[estimate, lower, upper]).reset_index(drop=True)
    if data.empty:
        pass
        return
    data["Display_Label"] = data[label_column].map(clean_label)
    data.to_csv(TABLE_DIR / f"{stem}.csv", index=False)
    fig, ax = plt.subplots(figsize=(9.2, max(3.8, 0.47 * len(data) + 1.8)))
    y = np.arange(len(data))
    values = data[estimate].to_numpy(float)
    lows = data[lower].to_numpy(float)
    highs = data[upper].to_numpy(float)
    ax.errorbar(values, y, xerr=[np.maximum(0, values-lows), np.maximum(0, highs-values)],
                fmt="o", color=color, ecolor=color, capsize=3, markersize=6.5, elinewidth=1.4)
    ax.axvline(0, color=CNS["gray"], lw=1.0)
    ax.set_yticks(y); ax.set_yticklabels(data["Display_Label"]); ax.invert_yaxis()
    ax.set_xlabel(f"IPCW {metric_label} estimate (95% CI)")
    ax.set_title(f"{horizon:g}-year {metric_label}", loc="left", fontweight="bold")
    clean_axis(ax, "x"); fig.tight_layout(); save_show(fig, stem)

FEATURE_ORDER = [
    "Clinical_only", "Clinical_plus_PRS", "Clinical_plus_NMR_score",
    "Clinical_plus_PRS_plus_NMR_score",
]
FEATURE_LABELS = {
    "Clinical_only": "Clinical",
    "Clinical_plus_PRS": "Clinical + PRS",
    "Clinical_plus_NMR_score": "Clinical + NMR_score",
    "Clinical_plus_PRS_plus_NMR_score": "Clinical + PRS + NMR_score",
}
FEATURE_COLORS = {
    "Clinical_only": CNS["gray"], "Clinical_plus_PRS": CNS["blue"],
    "Clinical_plus_NMR_score": CNS["green"],
    "Clinical_plus_PRS_plus_NMR_score": CNS["red"],
}

def load_calibration(horizon: float) -> pd.DataFrame:
    source = find_unique("result5_final_calibration_by_decile_km_ci.csv")
    data = pd.read_csv(source)
    require_columns(data, ["Feature_Set", "Horizon_Years", "Predicted_Risk", "Observed_Risk",
                           "Observed_Risk_Lower95", "Observed_Risk_Upper95"], source)
    mask = data["Feature_Set"].astype(str).eq("Clinical_plus_PRS_plus_NMR_score")
    mask &= np.isclose(pd.to_numeric(data["Horizon_Years"], errors="coerce"), float(horizon))
    result = data.loc[mask].sort_values("Predicted_Risk").copy()
    if result.empty:
        raise RuntimeError(f"{source} has no {horizon:g}-year calibration data for the joint model")
    return result

def load_dca(horizon: float) -> pd.DataFrame:
    source = find_unique("result5_final_standard_survival_dca.csv")
    data = pd.read_csv(source)
    require_columns(data, ["Horizon_Years", "Strategy", "Threshold", "Net_Benefit", "Feature_Set"], source)
    result = data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"], errors="coerce"), float(horizon))].copy()
    if result.empty:
        raise RuntimeError(f"{source} has no {horizon:g}-year DCA data")
    return result

def plot_calibration(horizon: float, stem: str):
    data = load_calibration(horizon)
    data.to_csv(TABLE_DIR / f"{stem}.csv", index=False)
    x = data["Predicted_Risk"].to_numpy(float) * 100
    y = data["Observed_Risk"].to_numpy(float) * 100
    lo = data["Observed_Risk_Lower95"].to_numpy(float) * 100
    hi = data["Observed_Risk_Upper95"].to_numpy(float) * 100
    axis_max = max(1.0, float(np.nanmax(np.r_[x, hi])) * 1.15)
    fig, ax = plt.subplots(figsize=(7.0, 5.4))
    ax.plot([0, axis_max], [0, axis_max], color=CNS["gray"], ls="--", lw=1.0, label="Ideal")
    ax.errorbar(x, y, yerr=np.vstack([np.maximum(0,y-lo), np.maximum(0,hi-y)]), fmt="o-",
                color=CNS["red"], ecolor=CNS["red"], lw=1.6, elinewidth=0.9,
                capsize=2.5, markersize=5.6, markeredgecolor="white", markeredgewidth=0.6,
                label="Observed KM risk (95% CI)")
    ax.set(xlim=(0,axis_max), ylim=(0,axis_max),
           xlabel=f"Predicted {horizon:g}-year CAD risk (%)",
           ylabel=f"Observed {horizon:g}-year CAD risk (%)")
    ax.set_title(f"{horizon:g}-year calibration (Cox PH)", loc="left", fontweight="bold")
    clean_axis(ax, "both"); ax.legend(frameon=False, loc="upper left")
    fig.tight_layout(); save_show(fig, stem)

def plot_observed_predicted(horizon: float, stem: str):
    data = load_calibration(horizon).reset_index(drop=True)
    data.to_csv(TABLE_DIR / f"{stem}.csv", index=False)
    x = np.arange(1, len(data)+1)
    obs = data["Observed_Risk"].to_numpy(float)*100
    pred = data["Predicted_Risk"].to_numpy(float)*100
    lo = data["Observed_Risk_Lower95"].to_numpy(float)*100
    hi = data["Observed_Risk_Upper95"].to_numpy(float)*100
    fig, ax = plt.subplots(figsize=(7.4, 5.3))
    ax.errorbar(x, obs, yerr=np.vstack([np.maximum(0,obs-lo), np.maximum(0,hi-obs)]),
                fmt="o-", color=CNS["red"], ecolor=CNS["red"], lw=1.7,
                elinewidth=0.8, capsize=2.5, markersize=5.3, label="Observed KM risk (95% CI)")
    ax.plot(x, pred, "s-", color=CNS["blue"], lw=1.5, markersize=4.8, label="Predicted")
    ax.set_xticks(x); ax.set_xlabel("Predicted-risk decile")
    ax.set_ylabel(f"{horizon:g}-year CAD risk (%)")
    ax.set_title(f"{horizon:g}-year observed versus predicted CAD risk", loc="left", fontweight="bold")
    clean_axis(ax, "y"); ax.legend(frameon=False, loc="upper left")
    fig.tight_layout(); save_show(fig, stem)

def plot_standard_dca(horizon: float, stem: str, xlim):
    data = load_dca(horizon)
    data.to_csv(TABLE_DIR / f"{stem}.csv", index=False)
    fig, ax = plt.subplots(figsize=(7.6, 5.5))
    order = list(FEATURE_LABELS.values()) + ["Treat all", "Treat none"]
    label_to_feature = {value:key for key,value in FEATURE_LABELS.items()}
    for strategy in order:
        use = data.loc[data["Strategy"].astype(str).eq(strategy)].sort_values("Threshold")
        if use.empty: continue
        if strategy == "Treat all": color, ls, lw = CNS["gray"], "--", 1.1
        elif strategy == "Treat none": color, ls, lw = CNS["gray"], ":", 1.1
        else:
            feature = label_to_feature[strategy]
            color, ls = FEATURE_COLORS[feature], "-"
            lw = 2.2 if feature == "Clinical_plus_PRS_plus_NMR_score" else 1.5
        ax.plot(use["Threshold"].astype(float)*100, use["Net_Benefit"].astype(float),
                color=color, ls=ls, lw=lw, label=strategy)
    ax.set_xlim(*xlim); ax.set_xlabel(f"{horizon:g}-year CAD risk threshold (%)")
    ax.set_ylabel("Net benefit")
    ax.set_title(f"{horizon:g}-year survival decision curve analysis", loc="left", fontweight="bold")
    clean_axis(ax, "both"); ax.legend(frameon=False, loc="best")
    fig.tight_layout(); save_show(fig, stem)

def plot_incremental_dca(horizon: float, stem: str, xlim):

    data = load_dca(horizon)
    model = data.loc[data["Feature_Set"].notna()].copy()
    pivot = model.pivot_table(index="Threshold", columns="Feature_Set", values="Net_Benefit", aggfunc="mean").sort_index()
    target = "Clinical_plus_PRS_plus_NMR_score"; reference = "Clinical_plus_PRS"
    if target not in pivot.columns or reference not in pivot.columns:
        raise RuntimeError(f"Missing DCA target or reference: {list(pivot.columns)}")
    out = pd.DataFrame({
        "Threshold": pivot.index.astype(float),
        "Target": target, "Reference": reference,
        "Delta_Net_Benefit": (pivot[target] - pivot[reference]).to_numpy(float),
    })
    out = out.loc[(out["Threshold"]*100 >= xlim[0]) & (out["Threshold"]*100 <= xlim[1])].copy()
    out.to_csv(TABLE_DIR / f"{stem}.csv", index=False)
    fig, ax = plt.subplots(figsize=(7.6, 5.5))
    ax.plot(out["Threshold"]*100, out["Delta_Net_Benefit"], color=CNS["red"], lw=2.3,
            label="Clinical + PRS + NMR_score vs Clinical + PRS")
    ax.axhline(0, color=CNS["gray"], ls=":", lw=1.0)
    ax.set_xlim(*xlim); ax.set_xlabel(f"{horizon:g}-year CAD risk threshold (%)")
    ax.set_ylabel("Difference in net benefit relative to Clinical + PRS")
    ax.set_title(f"{horizon:g}-year incremental clinical utility", loc="left", fontweight="bold")
    clean_axis(ax, "both"); ax.legend(frameon=False, loc="best")
    fig.tight_layout(); save_show(fig, stem)

PRS_ORDER = ["PRS_low", "PRS_mid", "PRS_high"]
NMR_ORDER = ["NMR_low", "NMR_mid", "NMR_high"]
PRS_LABELS = {"PRS_low":"Low PRS", "PRS_mid":"Intermediate PRS", "PRS_high":"High PRS"}
NMR_LABELS = {"NMR_low":"Low NMR_score", "NMR_mid":"Intermediate NMR_score", "NMR_high":"High NMR_score"}
GROUP_ORDER = [f"{prs}__{nmr}" for prs in PRS_ORDER for nmr in NMR_ORDER]
GROUP_LABELS = {f"{prs}__{nmr}": f"{PRS_LABELS[prs]} / {NMR_LABELS[nmr]}" for prs in PRS_ORDER for nmr in NMR_ORDER}
KM_COLORS = {
    "PRS_low__NMR_low":"#174A8B", "PRS_low__NMR_mid":"#7EA6D8", "PRS_low__NMR_high":"#24A6B5",
    "PRS_mid__NMR_low":"#F4A582", "PRS_mid__NMR_mid":"#E66B4D", "PRS_mid__NMR_high":"#D73027",
    "PRS_high__NMR_low":"#B8A1C9", "PRS_high__NMR_mid":"#7B3294", "PRS_high__NMR_high":"#A50F15",
}

print("RUN_ROOT:", RUN_ROOT)
print("Output:", OUTPUT_ROOT)

def ordered_values(data, column, preferred):
    seen = data[column].dropna().astype(str).tolist()
    return [x for x in preferred if x in seen] + [x for x in dict.fromkeys(seen) if x not in preferred]

def result6_frame(filename):
    source = find_unique(filename)
    data = pd.read_csv(source)
    require_columns(data, ["Metabolite_Label", "Beta", "HR", "HR_Lower95", "HR_Upper95", "FDR"], source)
    for column in ["Beta", "HR", "HR_Lower95", "HR_Upper95", "FDR"]:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    if "MinusLog10FDR" not in data:
        data["MinusLog10FDR"] = -np.log10(data["FDR"].clip(lower=np.finfo(float).tiny))
    if "Color_Meaning" not in data:
        data["Color_Meaning"] = np.where(
            data["FDR"] >= 0.05, "Not FDR-significant",
            np.where(data["Beta"] >= 0, "Positive coefficient (FDR < 0.05)", "Negative coefficient (FDR < 0.05)"),
        )
    color_map = {
        "Positive coefficient (FDR < 0.05)": CNS["red"],
        "Negative coefficient (FDR < 0.05)": CNS["blue"],
        "Not FDR-significant": CNS["gray"],
    }
    data["Color_Hex"] = data["Color_Meaning"].map(color_map).fillna(CNS["gray"])
    return data

def result6_volcano(
    data,
    title,
    xlabel,
    stem,
    n_labels_each_side=4,
):
    """
    Plot log hazard ratios against unadjusted P values and mark BH FDR significance.
    
    
    
    
    
    """

    from matplotlib.lines import Line2D

    data = data.copy()

    required = ["Metabolite_Label", "Beta", "P", "FDR"]
    missing = [
        column for column in required
        if column not in data.columns
    ]
    if missing:
        raise RuntimeError(
            f"{stem} missing required columns: {missing}"
        )

    for column in ["Beta", "P", "FDR"]:
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce",
        )

    data = data.dropna(
        subset=["Metabolite_Label", "Beta", "P", "FDR"]
    ).copy()

    duplicated = data.loc[
        data["Metabolite_Label"]
        .astype(str)
        .duplicated(False),
        "Metabolite_Label",
    ].astype(str).unique()

    if len(duplicated) > 0:
        raise RuntimeError(
            "Duplicate metabolites in the analysis table: "
            + ", ".join(duplicated[:12])
        )

    tiny = np.finfo(float).tiny

    data["P_for_plot"] = data["P"].clip(
        lower=tiny,
        upper=1.0,
    )
    data["MinusLog10P"] = -np.log10(
        data["P_for_plot"]
    )
    data["Abs_Beta"] = data["Beta"].abs()

    data["Plot_Group"] = np.select(
        [
            (data["FDR"] < 0.05) & (data["Beta"] < 0),
            (data["FDR"] < 0.05) & (data["Beta"] >= 0),
        ],
        [
            "Negative coefficient",
            "Positive coefficient",
        ],
        default="Not FDR-significant",
    )

    significant = data.loc[
        data["FDR"] < 0.05
    ].copy()

    negative_labels = (
        significant.loc[significant["Beta"] < 0]
        .sort_values(
            ["FDR", "P", "Abs_Beta"],
            ascending=[True, True, False],
        )
        .head(n_labels_each_side)
        .copy()
    )

    positive_labels = (
        significant.loc[significant["Beta"] >= 0]
        .sort_values(
            ["FDR", "P", "Abs_Beta"],
            ascending=[True, True, False],
        )
        .head(n_labels_each_side)
        .copy()
    )

    negative_labels["Plot_Number"] = np.arange(
        1,
        len(negative_labels) + 1,
    )

    positive_labels["Plot_Number"] = np.arange(
        len(negative_labels) + 1,
        len(negative_labels) + len(positive_labels) + 1,
    )

    labelled = pd.concat(
        [negative_labels, positive_labels],
        ignore_index=True,
    )

    number_map = dict(zip(
        labelled["Metabolite_Label"].astype(str),
        labelled["Plot_Number"].astype(int),
    ))

    data["Plot_Number"] = (
        data["Metabolite_Label"]
        .astype(str)
        .map(number_map)
    )

    data.to_csv(
        TABLE_DIR / f"{stem}_source_data.csv",
        index=False,
    )

    fig = plt.figure(figsize=(10.8, 6.3))

    grid = fig.add_gridspec(
        nrows=1,
        ncols=2,
        width_ratios=[4.1, 1.65],
        wspace=0.07,
    )

    ax = fig.add_subplot(grid[0, 0])
    key_ax = fig.add_subplot(grid[0, 1])
    key_ax.axis("off")

    colors = {
        "Not FDR-significant": "#BDBDBD",
        "Negative coefficient": CNS["blue"],
        "Positive coefficient": CNS["red"],
    }

    sizes = {
        "Not FDR-significant": 23,
        "Negative coefficient": 31,
        "Positive coefficient": 31,
    }

    alphas = {
        "Not FDR-significant": 0.48,
        "Negative coefficient": 0.88,
        "Positive coefficient": 0.88,
    }

    for group in [
        "Not FDR-significant",
        "Negative coefficient",
        "Positive coefficient",
    ]:
        part = data.loc[
            data["Plot_Group"].eq(group)
        ]

        ax.scatter(
            part["Beta"],
            part["MinusLog10P"],
            s=sizes[group],
            color=colors[group],
            alpha=alphas[group],
            edgecolor="white",
            linewidth=0.45,
            zorder=2,
        )

    for _, row in labelled.iterrows():
        point_color = (
            CNS["blue"]
            if row["Beta"] < 0
            else CNS["red"]
        )

        ax.scatter(
            [row["Beta"]],
            [row["MinusLog10P"]],
            s=125,
            facecolor=point_color,
            edgecolor=CNS["black"],
            linewidth=0.8,
            zorder=5,
        )

        ax.text(
            row["Beta"],
            row["MinusLog10P"],
            str(int(row["Plot_Number"])),
            ha="center",
            va="center",
            color="white",
            fontsize=7.4,
            fontweight="bold",
            zorder=6,
        )

    ax.axvline(
        0,
        color="#555555",
        linestyle=(0, (1.5, 2.0)),
        linewidth=1.0,
        zorder=1,
    )

    if not significant.empty:
        bh_p_boundary = float(
            significant["P"].max()
        )
        bh_y_boundary = -np.log10(
            max(bh_p_boundary, tiny)
        )

        ax.axhline(
            bh_y_boundary,
            color="#777777",
            linestyle=(0, (4, 3)),
            linewidth=1.0,
            zorder=1,
        )

        ax.annotate(
            "BH–FDR 5% boundary",
            xy=(0.985, bh_y_boundary),
            xycoords=(
                "axes fraction",
                "data",
            ),
            xytext=(0, 5),
            textcoords="offset points",
            ha="right",
            va="bottom",
            fontsize=8.5,
            color="#666666",
            bbox={
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.82,
                "pad": 1.0,
            },
        )

    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="none",
            markerfacecolor="#BDBDBD",
            markeredgecolor="white",
            markersize=6.5,
            label="FDR ≥ 0.05",
        ),
        Line2D(
            [0], [0],
            marker="o",
            linestyle="none",
            markerfacecolor=CNS["blue"],
            markeredgecolor="white",
            markersize=6.5,
            label="Negative, FDR < 0.05",
        ),
        Line2D(
            [0], [0],
            marker="o",
            linestyle="none",
            markerfacecolor=CNS["red"],
            markeredgecolor="white",
            markersize=6.5,
            label="Positive, FDR < 0.05",
        ),
    ]

    ax.legend(
        handles=legend_handles,
        frameon=False,
        loc="lower left",
        bbox_to_anchor=(0.0, 1.01),
        ncol=3,
        fontsize=8.8,
        columnspacing=1.15,
        handletextpad=0.35,
        borderaxespad=0,
    )

    key_ax.text(
        0.02,
        0.96,
        "Labelled signals",
        transform=key_ax.transAxes,
        fontsize=11.5,
        fontweight="bold",
        ha="left",
        va="top",
    )

    key_ax.plot(
        [0.02, 0.98],
        [0.925, 0.925],
        transform=key_ax.transAxes,
        color="#D8D8D8",
        linewidth=0.8,
        clip_on=False,
    )

    key_ax.text(
        0.02,
        0.885,
        "Negative coefficient",
        transform=key_ax.transAxes,
        color=CNS["blue"],
        fontsize=9.5,
        fontweight="bold",
        ha="left",
        va="top",
    )

    y_position = 0.835

    for _, row in negative_labels.iterrows():
        number = int(row["Plot_Number"])

        key_ax.text(
            0.04,
            y_position,
            str(number),
            transform=key_ax.transAxes,
            ha="center",
            va="center",
            fontsize=7.4,
            color="white",
            fontweight="bold",
            bbox={
                "boxstyle": "circle,pad=0.25",
                "facecolor": CNS["blue"],
                "edgecolor": CNS["black"],
                "linewidth": 0.6,
            },
        )

        key_ax.text(
            0.13,
            y_position,
            str(row["Metabolite_Label"]),
            transform=key_ax.transAxes,
            ha="left",
            va="center",
            fontsize=9.2,
            color=CNS["black"],
        )

        y_position -= 0.065

    y_position -= 0.025

    key_ax.text(
        0.02,
        y_position,
        "Positive coefficient",
        transform=key_ax.transAxes,
        color=CNS["red"],
        fontsize=9.5,
        fontweight="bold",
        ha="left",
        va="top",
    )

    y_position -= 0.055

    for _, row in positive_labels.iterrows():
        number = int(row["Plot_Number"])

        key_ax.text(
            0.04,
            y_position,
            str(number),
            transform=key_ax.transAxes,
            ha="center",
            va="center",
            fontsize=7.4,
            color="white",
            fontweight="bold",
            bbox={
                "boxstyle": "circle,pad=0.25",
                "facecolor": CNS["red"],
                "edgecolor": CNS["black"],
                "linewidth": 0.6,
            },
        )

        key_ax.text(
            0.13,
            y_position,
            str(row["Metabolite_Label"]),
            transform=key_ax.transAxes,
            ha="left",
            va="center",
            fontsize=9.2,
            color=CNS["black"],
        )

        y_position -= 0.065

    key_ax.text(
        0.02,
        0.055,
        "Labels show the four lowest-FDR\nsignals in each direction.",
        transform=key_ax.transAxes,
        fontsize=8.1,
        color="#666666",
        ha="left",
        va="bottom",
        linespacing=1.25,
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"$-\log_{10}$(raw $P$)")

    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.suptitle(
        title,
        x=0.075,
        y=0.985,
        ha="left",
        fontsize=15.5,
        fontweight="bold",
    )

    fig.subplots_adjust(
        left=0.08,
        right=0.98,
        top=0.84,
        bottom=0.14,
    )

    save_show(fig, stem)

def result6_forest(data, title, xlabel, stem, n=10):
    pool = data.loc[data["FDR"] < 0.05].copy()
    if pool.empty:
        pool = data.dropna(subset=["HR", "HR_Lower95", "HR_Upper95"]).copy()
    pool = pool.sort_values(["FDR", "Metabolite_Label"]).head(n).sort_values("Beta").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9.0, max(5.1, 0.46 * len(pool) + 1.9)))
    y = np.arange(len(pool))
    for pos, row in pool.iterrows():
        ax.errorbar(row["HR"], pos,
                    xerr=[[max(0, row["HR"]-row["HR_Lower95"])], [max(0, row["HR_Upper95"]-row["HR"])]],
                    fmt="o", color=row["Color_Hex"], ecolor=row["Color_Hex"], capsize=3, markersize=6)
    ax.axvline(1, color=CNS["black"], ls=":", lw=1); ax.set_xscale("log")
    ax.set_yticks(y); ax.set_yticklabels(pool["Metabolite_Label"]); ax.set_ylim(-0.7, len(pool)-0.3)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.2f}"))
    ax.set_xlabel(xlabel); ax.set_title(title, loc="left", fontweight="bold")
    clean_axis(ax, "x"); fig.tight_layout(); save_show(fig, stem)

def result6_interaction_forest(
    data,
    stem,
    n=10,
):
    """
    Plot the lowest-FDR interactions on a linear hazard-ratio scale.
    

    
    
    
    
    """

    from matplotlib.ticker import FormatStrFormatter

    data = data.copy()

    required = [
        "Metabolite_Label",
        "Beta",
        "HR",
        "HR_Lower95",
        "HR_Upper95",
        "FDR",
    ]
    missing = [
        column for column in required
        if column not in data.columns
    ]
    if missing:
        raise RuntimeError(
            f"Figure 7Dmissing required columns: {missing}"
        )

    for column in [
        "Beta",
        "HR",
        "HR_Lower95",
        "HR_Upper95",
        "FDR",
    ]:
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce",
        )

    data = data.dropna(
        subset=required
    ).copy()

    invalid_ci = (
        (data["HR_Lower95"] > data["HR"])
        | (data["HR_Upper95"] < data["HR"])
        | (data["HR_Lower95"] <= 0)
    )

    if invalid_ci.any():
        raise RuntimeError(
            "An HR lies outside its 95% confidence interval."
        )

    pool = (
        data.loc[data["FDR"] < 0.05]
        .sort_values(
            ["FDR", "Metabolite_Label"]
        )
        .head(n)
        .copy()
    )

    if pool.empty:
        raise RuntimeError(
            "No interactions passed FDR < 0.05."
        )

    pool["Selection_Rank_FDR"] = np.arange(
        1,
        len(pool) + 1,
    )

    pool = (
        pool.sort_values(
            ["HR", "FDR"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    pool["Direction"] = np.where(
        pool["Beta"] >= 0,
        "Positive",
        "Negative",
    )

    pool.to_csv(
        TABLE_DIR / f"{stem}_source_data.csv",
        index=False,
    )

    figure_height = max(
        5.3,
        0.48 * len(pool) + 1.8,
    )

    fig = plt.figure(
        figsize=(10.0, figure_height)
    )

    grid = fig.add_gridspec(
        1,
        2,
        width_ratios=[3.5, 1.65],
        wspace=0.03,
    )

    ax = fig.add_subplot(grid[0, 0])
    value_ax = fig.add_subplot(
        grid[0, 1],
        sharey=ax,
    )

    y = np.arange(len(pool))

    for position in y:
        if position % 2 == 0:
            for current_ax in [ax, value_ax]:
                current_ax.axhspan(
                    position - 0.5,
                    position + 0.5,
                    color="#F5F5F5",
                    zorder=0,
                )

    for position, row in pool.iterrows():
        point_color = (
            CNS["red"]
            if row["Beta"] >= 0
            else CNS["blue"]
        )

        ax.plot(
            [
                row["HR_Lower95"],
                row["HR_Upper95"],
            ],
            [position, position],
            color="#9A9A9A",
            linewidth=1.25,
            solid_capstyle="round",
            zorder=2,
        )

        cap_height = 0.075
        ax.plot(
            [
                row["HR_Lower95"],
                row["HR_Lower95"],
            ],
            [
                position - cap_height,
                position + cap_height,
            ],
            color="#9A9A9A",
            linewidth=1.0,
            zorder=2,
        )
        ax.plot(
            [
                row["HR_Upper95"],
                row["HR_Upper95"],
            ],
            [
                position - cap_height,
                position + cap_height,
            ],
            color="#9A9A9A",
            linewidth=1.0,
            zorder=2,
        )

        ax.scatter(
            row["HR"],
            position,
            s=58,
            facecolor=point_color,
            edgecolor="white",
            linewidth=0.7,
            zorder=4,
        )

    ax.axvline(
        1.0,
        color="#444444",
        linestyle=(0, (2, 2)),
        linewidth=1.0,
        zorder=1,
    )

    ci_min = float(pool["HR_Lower95"].min())
    ci_max = float(pool["HR_Upper95"].max())
    ci_span = max(ci_max - ci_min, 0.01)

    x_lower = min(
        0.995,
        ci_min - 0.06 * ci_span,
    )
    x_upper = ci_max + 0.06 * ci_span

    ax.set_xlim(x_lower, x_upper)

    tick_start = np.floor(x_lower / 0.01) * 0.01
    tick_end = np.ceil(x_upper / 0.01) * 0.01

    ticks = np.arange(
        tick_start,
        tick_end + 0.005,
        0.02,
    )

    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(
        FormatStrFormatter("%.2f")
    )

    ax.set_yticks(y)
    ax.set_yticklabels(
        pool["Metabolite_Label"],
        fontsize=10.5,
    )

    ax.set_ylim(
        len(pool) - 0.5,
        -0.5,
    )

    ax.set_xlabel(
        "Interaction hazard ratio "
        "(per 1-SD PRS × 1-SD metabolite)"
    )

    value_ax.set_xlim(0, 1)
    value_ax.set_ylim(
        len(pool) - 0.5,
        -0.5,
    )

    value_ax.text(
        0.03,
        1.025,
        "HR (95% CI)",
        transform=value_ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=9.7,
        fontweight="bold",
    )

    value_ax.text(
        0.98,
        1.025,
        "FDR",
        transform=value_ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=9.7,
        fontweight="bold",
    )

    def format_fdr(value):
        if value < 0.001:
            return f"{value:.1e}"
        return f"{value:.3f}"

    for position, row in pool.iterrows():
        hr_text = (
            f'{row["HR"]:.3f} '
            f'({row["HR_Lower95"]:.3f}–'
            f'{row["HR_Upper95"]:.3f})'
        )

        value_ax.text(
            0.03,
            position,
            hr_text,
            ha="left",
            va="center",
            fontsize=9.3,
            color=CNS["black"],
        )

        value_ax.text(
            0.98,
            position,
            format_fdr(row["FDR"]),
            ha="right",
            va="center",
            fontsize=9.1,
            color="#555555",
        )

    value_ax.tick_params(
        left=False,
        labelleft=False,
        bottom=False,
        labelbottom=False,
    )

    for spine in value_ax.spines.values():
        spine.set_visible(False)

    ax.set_title(
        "Top PRS–metabolite interaction signals",
        loc="left",
        fontsize=15,
        fontweight="bold",
        pad=16,
    )

    ax.text(
        0,
        1.015,
        "Selected by lowest FDR and displayed in descending HR order",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=9.3,
        color="#666666",
    )

    ax.grid(
        axis="x",
        color="#E3E3E3",
        linewidth=0.65,
        alpha=0.75,
    )
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.subplots_adjust(
        left=0.18,
        right=0.98,
        top=0.88,
        bottom=0.14,
    )

    save_show(
        fig,
        stem,
    )


In [ ]:
try:
    source = find_unique("figure1A_cohort_flow.csv")
    data = pd.read_csv(source); require_columns(data, ["Step", "N"], source)
    fig, ax = plt.subplots(figsize=(11.5, max(6.0, 0.78*len(data)+1.4))); ax.axis("off")
    colors = [CNS["blue"], CNS["teal"], CNS["green"], CNS["orange"], CNS["purple"], CNS["red"]]
    y = np.linspace(0.94, 0.06, len(data))
    for index, (position, row) in enumerate(zip(y, data.to_dict("records"))):
        ax.text(0.5, position, f"{row['Step']}\nN = {int(row['N']):,}", ha="center", va="center", fontsize=12.5,
                bbox=dict(boxstyle="round,pad=0.45", facecolor=colors[index%len(colors)], alpha=0.13,
                          edgecolor=colors[index%len(colors)], linewidth=1.5))
        if index < len(data)-1:
            ax.annotate("", xy=(0.5, y[index+1]+0.025), xytext=(0.5, position-0.025),
                        arrowprops=dict(arrowstyle="->", color=CNS["gray"], lw=1.5))
    fig.tight_layout(); save_show(fig, "figure1A_cohort_flow_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    source = find_unique("figure1B_analysis_workflow.csv")
    data = pd.read_csv(source); require_columns(data, ["Stage", "Definition"], source)
    fig, ax = plt.subplots(figsize=(15.5, 4.5)); ax.axis("off")
    colors = [CNS["blue"], CNS["teal"], CNS["green"], CNS["orange"], CNS["purple"], CNS["red"]]
    x = np.linspace(0.08, 0.92, len(data))
    for index, (position, row) in enumerate(zip(x, data.to_dict("records"))):
        ax.text(position, 0.53, f"{row['Stage']}\n{row['Definition']}", ha="center", va="center", fontsize=11.5, wrap=True,
                bbox=dict(boxstyle="round,pad=0.5", facecolor=colors[index%len(colors)], alpha=0.13,
                          edgecolor=colors[index%len(colors)], linewidth=1.5))
        if index < len(data)-1:
            ax.annotate("", xy=(x[index+1]-0.075, 0.53), xytext=(position+0.075, 0.53),
                        arrowprops=dict(arrowstyle="->", color=CNS["gray"], lw=1.5))
    fig.tight_layout(); save_show(fig, "figure1B_analysis_workflow_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Included versus excluded
try:
    source = find_unique('Supplementary_SMD_included_vs_excluded.csv')
    data = pd.read_csv(source); require_columns(data, ["Variable", "SMD"], source)
    data["SMD"] = pd.to_numeric(data["SMD"], errors="coerce")
    data = data.dropna(subset=["SMD"]).assign(Abs_SMD=lambda x: x["SMD"].abs()).sort_values("Abs_SMD")
    fig, ax = plt.subplots(figsize=(9.5, max(5.5, 0.48*len(data))))
    ax.barh(data["Variable"], data["SMD"], color=np.where(data["SMD"]>=0, CNS["blue"], CNS["red"]))
    ax.axvline(0, color=CNS["black"], lw=1); ax.axvline(0.1, color=CNS["gray"], ls="--"); ax.axvline(-0.1, color=CNS["gray"], ls="--")
    ax.set_xlabel("Standardized mean difference"); ax.set_title('Included versus excluded', loc="left", fontweight="bold")
    clean_axis(ax, "x"); fig.tight_layout(); save_show(fig, 'supplementary_smd_included_vs_excluded')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Incident CAD versus no incident CAD
try:
    source = find_unique('Supplementary_SMD_event_vs_nonevent.csv')
    data = pd.read_csv(source); require_columns(data, ["Variable", "SMD"], source)
    data["SMD"] = pd.to_numeric(data["SMD"], errors="coerce")
    data = data.dropna(subset=["SMD"]).assign(Abs_SMD=lambda x: x["SMD"].abs()).sort_values("Abs_SMD")
    fig, ax = plt.subplots(figsize=(9.5, max(5.5, 0.48*len(data))))
    ax.barh(data["Variable"], data["SMD"], color=np.where(data["SMD"]>=0, CNS["blue"], CNS["red"]))
    ax.axvline(0, color=CNS["black"], lw=1); ax.axvline(0.1, color=CNS["gray"], ls="--"); ax.axvline(-0.1, color=CNS["gray"], ls="--")
    ax.set_xlabel("Standardized mean difference"); ax.set_title('Incident CAD versus no incident CAD', loc="left", fontweight="bold")
    clean_axis(ax, "x"); fig.tight_layout(); save_show(fig, 'supplementary_smd_event_vs_nonevent')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Validation versus train
try:
    source = find_unique('Supplementary_SMD_split_balance.csv')
    data = pd.read_csv(source); require_columns(data, ["Variable", "SMD"], source)
    data = data.loc[data["Comparison"].astype(str).eq('Validation vs Train')].copy()
    data["SMD"] = pd.to_numeric(data["SMD"], errors="coerce")
    data = data.dropna(subset=["SMD"]).assign(Abs_SMD=lambda x: x["SMD"].abs()).sort_values("Abs_SMD")
    fig, ax = plt.subplots(figsize=(9.5, max(5.5, 0.48*len(data))))
    ax.barh(data["Variable"], data["SMD"], color=np.where(data["SMD"]>=0, CNS["blue"], CNS["red"]))
    ax.axvline(0, color=CNS["black"], lw=1); ax.axvline(0.1, color=CNS["gray"], ls="--"); ax.axvline(-0.1, color=CNS["gray"], ls="--")
    ax.set_xlabel("Standardized mean difference"); ax.set_title('Validation versus train', loc="left", fontweight="bold")
    clean_axis(ax, "x"); fig.tight_layout(); save_show(fig, 'supplementary_smd_validation_vs_train')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Test versus train
try:
    source = find_unique('Supplementary_SMD_split_balance.csv')
    data = pd.read_csv(source); require_columns(data, ["Variable", "SMD"], source)
    data = data.loc[data["Comparison"].astype(str).eq('Test vs Train')].copy()
    data["SMD"] = pd.to_numeric(data["SMD"], errors="coerce")
    data = data.dropna(subset=["SMD"]).assign(Abs_SMD=lambda x: x["SMD"].abs()).sort_values("Abs_SMD")
    fig, ax = plt.subplots(figsize=(9.5, max(5.5, 0.48*len(data))))
    ax.barh(data["Variable"], data["SMD"], color=np.where(data["SMD"]>=0, CNS["blue"], CNS["red"]))
    ax.axvline(0, color=CNS["black"], lw=1); ax.axvline(0.1, color=CNS["gray"], ls="--"); ax.axvline(-0.1, color=CNS["gray"], ls="--")
    ax.set_xlabel("Standardized mean difference"); ax.set_title('Test versus train', loc="left", fontweight="bold")
    clean_axis(ax, "x"); fig.tight_layout(); save_show(fig, 'supplementary_smd_test_vs_train')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    source = find_unique("figure2A_delta_uno_cindex.csv")
    data = pd.read_csv(source); require_columns(data, ["Comparison","Horizon_Years","Delta_Uno_C","Delta_Uno_C_Lower95","Delta_Uno_C_Upper95"], source)
    preferred = ["Clinical + PRS vs Clinical", "Clinical + NMR_score vs Clinical", "Clinical + PRS + NMR_score vs Clinical", "Integrated vs Clinical + PRS", "Integrated vs Clinical + NMR_score"]
    order = ordered_values(data, "Comparison", preferred); y = np.arange(len(order))
    fig, ax = plt.subplots(figsize=(9.5, 5.8))
    for horizon, color, marker, offset in [(5.0,CNS["teal"],"o",-0.11),(10.0,CNS["blue"],"s",0.11)]:
        part=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),horizon)].copy(); part["Order"]=part["Comparison"].map({x:i for i,x in enumerate(order)}); part=part.sort_values("Order")
        value=part["Delta_Uno_C"].to_numpy(float); lo=part["Delta_Uno_C_Lower95"].to_numpy(float); hi=part["Delta_Uno_C_Upper95"].to_numpy(float)
        ax.errorbar(value,y+offset,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt=marker,color=color,ecolor=color,capsize=3,markersize=6.5,label=f"{horizon:g} years")
    ax.axvline(0,color=CNS["gray"],lw=1,linestyle='--'); ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis()
    ax.set_xlabel("Paired delta Uno IPCW C-index (95% CI)"); ax.set_title("Incremental discrimination",loc="left",fontweight="bold")
    clean_axis(ax,"x"); ax.legend(frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5)); fig.tight_layout(); save_show(fig,"figure2A_delta_uno_ipcw_cindex_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
from matplotlib.patches import FancyBboxPatch

source = find_unique("figure2B_joint_model_adjusted_hr.csv")
data = pd.read_csv(source)

require_columns(
    data,
    ["Variable", "HR", "HR_CI_Lower", "HR_CI_Upper"],
    source,
)

data["Variable"] = data["Variable"].astype(str)

label_map = {
    "26227": "CAD polygenic risk score",
    "NMR_score": "NMR score",
}

subtitle_map = {
    "26227": "Per 1-SD increment",
    "NMR_score": "Per 1-SD increment",
}

color_map = {
    "26227": CNS["blue"],
    "NMR_score": CNS["green"],
}

data["Display_Label"] = (
    data["Variable"]
    .map(label_map)
    .fillna(data["Variable"])
)

data["Order"] = (
    data["Variable"]
    .map({"26227": 0, "NMR_score": 1})
    .fillna(9)
)

data = (
    data.sort_values("Order")
    .reset_index(drop=True)
)

all_lower = pd.to_numeric(
    data["HR_CI_Lower"],
    errors="coerce",
).to_numpy(float)

all_upper = pd.to_numeric(
    data["HR_CI_Upper"],
    errors="coerce",
).to_numpy(float)

scale_min = min(
    0.98,
    float(np.nanmin(all_lower)) * 0.98,
)

scale_max = max(
    1.05,
    float(np.nanmax(all_upper)) * 1.04,
)

def scale_hr(value, left, right):
    """Map hazard ratios to a shared log scale."""
    numerator = np.log(float(value)) - np.log(scale_min)
    denominator = np.log(scale_max) - np.log(scale_min)
    return left + numerator / denominator * (right - left)

def format_p(value):
    if pd.isna(value):
        return ""
    value = float(value)

    if value < 0.001:
        return "P < 0.001"

    return f"P = {value:.3f}"

fig, ax = plt.subplots(figsize=(9.4, 4.6))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

card_centers = [0.27, 0.73]
card_width = 0.40
card_bottom = 0.13
card_height = 0.72

for center, (_, row) in zip(
    card_centers,
    data.iterrows(),
):
    variable = row["Variable"]
    color = color_map.get(variable, CNS["blue"])

    estimate = float(row["HR"])
    lower = float(row["HR_CI_Lower"])
    upper = float(row["HR_CI_Upper"])

    left = center - card_width / 2
    right = center + card_width / 2

    card = FancyBboxPatch(
        (left, card_bottom),
        card_width,
        card_height,
        boxstyle="round,pad=0.018,rounding_size=0.025",
        linewidth=1.5,
        edgecolor=color,
        facecolor=color,
        alpha=0.08,
    )

    ax.add_patch(card)

    ax.text(
        center,
        0.76,
        row["Display_Label"],
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        color=CNS["black"],
    )

    ax.text(
        center,
        0.68,
        subtitle_map.get(variable, "Per 1-SD increment"),
        ha="center",
        va="center",
        fontsize=10.5,
        color=CNS["gray"],
    )

    ax.text(
        center,
        0.53,
        f"HR  {estimate:.2f}",
        ha="center",
        va="center",
        fontsize=22,
        fontweight="bold",
        color=color,
    )

    # 95%CI
    ax.text(
        center,
        0.42,
        f"95% CI  {lower:.2f}–{upper:.2f}",
        ha="center",
        va="center",
        fontsize=12,
        color=CNS["black"],
    )

    p_column = (
        "p"
        if "p" in data.columns
        else "P_Value"
        if "P_Value" in data.columns
        else None
    )

    if p_column is not None:
        ax.text(
            center,
            0.34,
            format_p(row[p_column]),
            ha="center",
            va="center",
            fontsize=10.5,
            color=CNS["gray"],
        )

    ci_left = left + 0.055
    ci_right = right - 0.055
    ci_y = 0.235

    ax.plot(
        [ci_left, ci_right],
        [ci_y, ci_y],
        color=CNS["light_gray"],
        linewidth=2,
        solid_capstyle="round",
    )

    null_x = scale_hr(
        1.0,
        ci_left,
        ci_right,
    )

    lower_x = scale_hr(
        lower,
        ci_left,
        ci_right,
    )

    estimate_x = scale_hr(
        estimate,
        ci_left,
        ci_right,
    )

    upper_x = scale_hr(
        upper,
        ci_left,
        ci_right,
    )

    ax.plot(
        [null_x, null_x],
        [ci_y - 0.035, ci_y + 0.035],
        color=CNS["gray"],
        linestyle="--",
        linewidth=1,
    )

    # 95%CI
    ax.plot(
        [lower_x, upper_x],
        [ci_y, ci_y],
        color=color,
        linewidth=3,
        solid_capstyle="round",
    )

    ax.plot(
        [lower_x, lower_x],
        [ci_y - 0.020, ci_y + 0.020],
        color=color,
        linewidth=1.5,
    )

    ax.plot(
        [upper_x, upper_x],
        [ci_y - 0.020, ci_y + 0.020],
        color=color,
        linewidth=1.5,
    )

    ax.scatter(
        estimate_x,
        ci_y,
        s=80,
        color=color,
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )

    ax.text(
        null_x,
        0.18,
        "HR=1",
        ha="center",
        va="center",
        fontsize=8.5,
        color=CNS["gray"],
    )

ax.set_title(
    "Independent contributions of genetic and metabolic risk",
    loc="left",
    fontsize=15,
    fontweight="bold",
    pad=10,
)

ax.text(
    0.5,
    0.035,
    "Mutually adjusted estimates from the same clinical + PRS + NMR score Cox model",
    ha="center",
    va="center",
    fontsize=10,
    color=CNS["gray"],
)

fig.tight_layout()

save_show(
    fig,
    "figure2B_joint_model_effect_cards",
)


In [ ]:
try:
    source=find_unique("figure2C_delta_ipcw_ibs.csv"); data=pd.read_csv(source)
    require_columns(data,["Comparison","Horizon_Years","Delta_IBS","Delta_IBS_Lower95","Delta_IBS_Upper95"],source)
    preferred=["Clinical + PRS vs Clinical","Clinical + NMR_score vs Clinical","Clinical + PRS + NMR_score vs Clinical","Integrated vs Clinical + PRS","Integrated vs Clinical + NMR_score"]
    order=ordered_values(data,"Comparison",preferred); y=np.arange(len(order)); scale=1e5 if np.nanmax(np.abs(data[["Delta_IBS","Delta_IBS_Lower95","Delta_IBS_Upper95"]].to_numpy(float)))<1e-3 else 1.0
    fig,ax=plt.subplots(figsize=(9.5,5.8))
    for horizon,color,marker,offset in [(5.0,CNS["teal"],"o",-0.11),(10.0,CNS["blue"],"s",0.11)]:
        part=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),horizon)].copy(); part["Order"]=part["Comparison"].map({x:i for i,x in enumerate(order)}); part=part.sort_values("Order")
        value=part["Delta_IBS"].to_numpy(float)*scale; lo=part["Delta_IBS_Lower95"].to_numpy(float)*scale; hi=part["Delta_IBS_Upper95"].to_numpy(float)*scale
        ax.errorbar(value,y+offset,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt=marker,color=color,ecolor=color,capsize=3,markersize=6.5,label=f"{horizon:g} years")
    ax.axvline(0,color=CNS["gray"],lw=1,linestyle='--'); ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis()
    ax.set_xlabel("Delta IPCW IBS: new minus reference"+(" (×10⁻⁵)" if scale==1e5 else "")+"; lower is better")
    ax.set_title("Prediction error",loc="left",fontweight="bold"); clean_axis(ax,"x"); ax.legend(frameon=False)
    fig.tight_layout(); save_show(fig,"figure2C_delta_ipcw_ibs_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure2D_ipcw_nri_idi.csv',
        horizon=5, metric='continuous_nri', label_column='Comparison',
        stem='figure2D_revised_5y_continuous_nri',
        filters={'Comparison_Group': 'Extension vs Clinical'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure2D_ipcw_nri_idi.csv',
        horizon=5, metric='categorical_nri', label_column='Comparison',
        stem='figure2D_revised_5y_categorical_nri',
        filters={'Comparison_Group': 'Extension vs Clinical'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure2D_ipcw_nri_idi.csv',
        horizon=5, metric='idi', label_column='Comparison',
        stem='figure2D_revised_5y_idi',
        filters={'Comparison_Group': 'Extension vs Clinical'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure2D_ipcw_nri_idi.csv',
        horizon=10, metric='continuous_nri', label_column='Comparison',
        stem='figure2D_revised_10y_continuous_nri',
        filters={'Comparison_Group': 'Extension vs Clinical'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure2D_ipcw_nri_idi.csv',
        horizon=10, metric='categorical_nri', label_column='Comparison',
        stem='figure2D_revised_10y_categorical_nri',
        filters={'Comparison_Group': 'Extension vs Clinical'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure2D_ipcw_nri_idi.csv',
        horizon=10, metric='idi', label_column='Comparison',
        stem='figure2D_revised_10y_idi',
        filters={'Comparison_Group': 'Extension vs Clinical'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Supplementary Result 2 — 5-year calibration
try:
    source=find_unique("supplement_result2_calibration_by_decile.csv"); data=pd.read_csv(source)
    require_columns(data,["Feature_Set","Horizon_Years","Predicted_Risk","Observed_KM_Risk"],source)
    data=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),5.0)].copy()
    labels={"Clinical_only":"Clinical","Clinical_plus_PRS":"Clinical + PRS","Clinical_plus_NMR_score":"Clinical + NMR_score","Clinical_plus_PRS_plus_NMR_score":"Clinical + PRS + NMR_score"}
    colors={"Clinical_only":CNS["black"],"Clinical_plus_PRS":CNS["blue"],"Clinical_plus_NMR_score":CNS["green"],"Clinical_plus_PRS_plus_NMR_score":CNS["red"]}
    fig,ax=plt.subplots(figsize=(8.4,6.4))
    for feature in labels:
        part=data.loc[data["Feature_Set"].astype(str).eq(feature)].sort_values("Predicted_Risk")
        if not part.empty: ax.plot(part["Predicted_Risk"],part["Observed_KM_Risk"],marker="o",lw=2,color=colors[feature],label=labels[feature])
    limit=max(0.01,float(data[["Predicted_Risk","Observed_KM_Risk"]].to_numpy(float).max()*1.05)); ax.plot([0,limit],[0,limit],"--",color=CNS["gray"],label="Ideal")
    ax.set_xlim(0,limit); ax.set_ylim(0,limit); ax.set_xlabel("Mean predicted CAD risk"); ax.set_ylabel("Kaplan–Meier observed CAD risk")
    ax.set_title("5-year calibration",loc="left",fontweight="bold"); clean_axis(ax,"both"); ax.legend(frameon=False)
    fig.tight_layout(); save_show(fig,"supplement_result2_5y_calibration_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Supplementary Result 2 — 10-year calibration
try:
    source=find_unique("supplement_result2_calibration_by_decile.csv"); data=pd.read_csv(source)
    require_columns(data,["Feature_Set","Horizon_Years","Predicted_Risk","Observed_KM_Risk"],source)
    data=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),10.0)].copy()
    labels={"Clinical_only":"Clinical","Clinical_plus_PRS":"Clinical + PRS","Clinical_plus_NMR_score":"Clinical + NMR_score","Clinical_plus_PRS_plus_NMR_score":"Clinical + PRS + NMR_score"}
    colors={"Clinical_only":CNS["black"],"Clinical_plus_PRS":CNS["blue"],"Clinical_plus_NMR_score":CNS["green"],"Clinical_plus_PRS_plus_NMR_score":CNS["red"]}
    fig,ax=plt.subplots(figsize=(8.4,6.4))
    for feature in labels:
        part=data.loc[data["Feature_Set"].astype(str).eq(feature)].sort_values("Predicted_Risk")
        if not part.empty: ax.plot(part["Predicted_Risk"],part["Observed_KM_Risk"],marker="o",lw=2,color=colors[feature],label=labels[feature])
    limit=max(0.01,float(data[["Predicted_Risk","Observed_KM_Risk"]].to_numpy(float).max()*1.05)); ax.plot([0,limit],[0,limit],"--",color=CNS["gray"],label="Ideal")
    ax.set_xlim(0,limit); ax.set_ylim(0,limit); ax.set_xlabel("Mean predicted CAD risk"); ax.set_ylabel("Kaplan–Meier observed CAD risk")
    ax.set_title("10-year calibration",loc="left",fontweight="bold"); clean_axis(ax,"both"); ax.legend(frameon=False)
    fig.tight_layout(); save_show(fig,"supplement_result2_10y_calibration_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 2E — 5-year survival DCA
try:
    source=find_unique("figure2E_survival_dca.csv"); data=pd.read_csv(source)
    require_columns(data,["Horizon_Years","Strategy","Threshold","Net_Benefit"],source); data=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),5.0)].copy()
    colors={"Clinical":CNS["black"],"Clinical + PRS":CNS["blue"],"Clinical + NMR_score":CNS["green"],"Clinical + PRS + NMR_score":CNS["red"],"Treat all":CNS["gray"],"Treat none":CNS["gray"]}
    fig,ax=plt.subplots(figsize=(7.8,5.7))
    for strategy,part in data.groupby("Strategy",sort=False):
        part=part.sort_values("Threshold"); ax.plot(part["Threshold"].astype(float)*100,part["Net_Benefit"],lw=2 if strategy=="Clinical + PRS + NMR_score" else 1.5,ls="--" if strategy=="Treat all" else (":" if strategy=="Treat none" else "-"),color=colors.get(strategy,CNS["gray"]),label=strategy)
    ax.set_xlabel("5-year CAD risk threshold (%)"); ax.set_ylabel("Net benefit"); ax.set_title("5-year survival decision curve analysis",loc="left",fontweight="bold")
    clean_axis(ax,"both"); ax.legend(frameon=False,fontsize=10); fig.tight_layout(); save_show(fig,"figure2E_5y_survival_dca_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 2E — 10-year survival DCA
try:
    source=find_unique("figure2E_survival_dca.csv"); data=pd.read_csv(source)
    require_columns(data,["Horizon_Years","Strategy","Threshold","Net_Benefit"],source); data=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),10.0)].copy()
    colors={"Clinical":CNS["black"],"Clinical + PRS":CNS["blue"],"Clinical + NMR_score":CNS["green"],"Clinical + PRS + NMR_score":CNS["red"],"Treat all":CNS["gray"],"Treat none":CNS["gray"]}
    fig,ax=plt.subplots(figsize=(7.8,5.7))
    for strategy,part in data.groupby("Strategy",sort=False):
        part=part.sort_values("Threshold"); ax.plot(part["Threshold"].astype(float)*100,part["Net_Benefit"],lw=2 if strategy=="Clinical + PRS + NMR_score" else 1.5,ls="--" if strategy=="Treat all" else (":" if strategy=="Treat none" else "-"),color=colors.get(strategy,CNS["gray"]),label=strategy)
    ax.set_xlabel("10-year CAD risk threshold (%)"); ax.set_ylabel("Net benefit"); ax.set_title("10-year survival decision curve analysis",loc="left",fontweight="bold")
    clean_axis(ax,"both"); ax.legend(frameon=False,fontsize=10); fig.tight_layout(); save_show(fig,"figure2E_10y_survival_dca_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Supplementary Result 2 — Harrell C-index sensitivity
try:
    source=find_unique("supplement_delta_harrell_cindex.csv"); data=pd.read_csv(source)
    require_columns(data,["Comparison","Delta_Harrell_C","Delta_Harrell_C_Lower95","Delta_Harrell_C_Upper95"],source)
    data=data.reset_index(drop=True); y=np.arange(len(data)); value=data["Delta_Harrell_C"].to_numpy(float); lo=data["Delta_Harrell_C_Lower95"].to_numpy(float); hi=data["Delta_Harrell_C_Upper95"].to_numpy(float)
    fig,ax=plt.subplots(figsize=(9,5.5)); ax.errorbar(value,y,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt="o",color=CNS["purple"],ecolor=CNS["purple"],capsize=3)
    ax.axvline(0,color=CNS["gray"],lw=1); ax.set_yticks(y); ax.set_yticklabels(data["Comparison"]); ax.invert_yaxis(); ax.set_xlabel("Paired delta Harrell C-index (95% CI)")
    ax.set_title("Harrell C-index sensitivity analysis",loc="left",fontweight="bold"); clean_axis(ax,"x"); fig.tight_layout(); save_show(fig,"supplement_result2_delta_harrell_cindex_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    scale_source = find_unique("figure3A_representation_scale.csv")
    scale = pd.read_csv(scale_source)
    require_columns(scale, ["Feature_Set", "NMR_Predictor_N"], scale_source)
    scale_features = [
        "Clinical_plus_PRS_plus_NMR_score",
        "Clinical_plus_PRS_plus_SpearmanNMR",
        "Clinical_plus_PRS_plus_Raw106NMR",
    ]
    scale = scale.loc[scale["Feature_Set"].isin(scale_features)].copy()
    if scale["Feature_Set"].duplicated().any() or set(scale["Feature_Set"]) != set(scale_features):
        raise ValueError("Missing or duplicate NMR representation dimensions")
    counts = pd.to_numeric(
        scale.set_index("Feature_Set").loc[scale_features, "NMR_Predictor_N"],
        errors="raise",
    ).to_numpy(dtype=float)
    if not np.isfinite(counts).all() or np.any(counts <= 0) or np.any(counts != np.floor(counts)):
        raise ValueError("NMR representation dimensions must be positive integers")
    result3_scale = pd.DataFrame({
        "Representation": ["NMR score", "Spearman-pruned", "Raw106"],
        "Final_Model_NMR_Input_Dimension": counts.astype(int),
    })
    score_manifest = json.loads((RUN_ROOT / "01_cohort" / "pipeline_manifest.json").read_text())
    score_panel = score_manifest["nmr_score"]["predictor_contract_name"]

    result3_scale.to_csv(
        TABLE_DIR / "figure3A_revised_representation_dimension.csv",
        index=False,
    )

    fig, ax = plt.subplots(
        figsize=(8.2, 5.2)
    )

    bars = ax.bar(
        result3_scale["Representation"],
        result3_scale["Final_Model_NMR_Input_Dimension"],
        color=[
            CNS["green"],
            CNS["purple"],
            CNS["red"],
        ],
        width=0.62,
        edgecolor="none",
    )

    for bar, value in zip(
        bars,
        result3_scale["Final_Model_NMR_Input_Dimension"],
    ):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 2.2,
            str(value),
            ha="center",
            va="bottom",
            fontsize=12.5,
            fontweight="bold",
        )

    ax.set_ylabel(
        "NMR predictors in final Cox model (n)",
        labelpad=10,
    )

    ax.set_title(
        "NMR representation dimensionality",
        loc="left",
        fontweight="bold",
        pad=12,
    )

    ax.set_ylim(0, 120)
    ax.set_yticks(np.arange(0, 121, 20))

    ax.text(
        0.0,
        -0.22,
        "NMR score entered the final model as one derived predictor,\n"
        f"and was developed from the {score_panel} candidate biomarker panel.",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10.2,
        color="#555555",
        linespacing=1.35,
    )

    ax.grid(
        axis="y",
        color=CNS["light_gray"],
        linewidth=0.7,
        alpha=0.70,
    )
    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.subplots_adjust(
        left=0.17,
        right=0.98,
        top=0.88,
        bottom=0.28,
    )

    save_show(
        fig,
        "figure3A_revised_final_model_input_dimension",
    )

except FileNotFoundError as error:
    print(
        "Figure 3A could not be drawn: ",
        error,
    )


In [ ]:
# Figure 3B — primary Cox analysis
try:
    source=find_unique('figure3B_delta_uno_cindex_vs_clinical_prs.csv'); data=pd.read_csv(source)
    data=data.loc[data["Algorithm"].astype(str).eq('Cox PH')].copy()
    require_columns(data,["Representation","Horizon_Years",'Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95'],source)
    preferred=["NMR_score","Spearman-pruned NMR","Raw106 NMR"]; order=ordered_values(data,"Representation",preferred); y=np.arange(len(order))
    scale=1e5 if 'Delta_Uno_C'=="Delta_IBS" and np.nanmax(np.abs(data[['Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95']].to_numpy(float)))<1e-3 else 1.0
    fig,ax=plt.subplots(figsize=(9.2,5.4))
    for horizon,color,marker,offset in [(5.0,CNS["teal"],"o",-0.11),(10.0,CNS["blue"],"s",0.11)]:
        part=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),horizon)].copy(); part["Order"]=part["Representation"].map({x:i for i,x in enumerate(order)}); part=part.sort_values("Order")
        value=part['Delta_Uno_C'].to_numpy(float)*scale; lo=part['Delta_Uno_C_Lower95'].to_numpy(float)*scale; hi=part['Delta_Uno_C_Upper95'].to_numpy(float)*scale
        ax.errorbar(value,y+offset,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt=marker,color=color,ecolor=color,capsize=3,markersize=6.5,label=f"{horizon:g} years")
    ax.axvline(0,color=CNS["gray"],lw=1,linestyle='--'); ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlabel('Delta Uno IPCW C-index vs Clinical + PRS (95% CI)'+(" (×10⁻⁵)" if scale==1e5 else ""))
    ax.set_title('Figure 3B — primary Cox analysis',loc="left",fontweight="bold"); clean_axis(ax,"x"); ax.legend(frameon=False); fig.tight_layout(); save_show(fig,'figure3B_cox_primary_delta_uno_plot_only')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3B — Elastic-net sensitivity
try:
    source=find_unique('figure3B_delta_uno_cindex_vs_clinical_prs.csv'); data=pd.read_csv(source)
    data=data.loc[data["Algorithm"].astype(str).eq('Elastic-net Cox')].copy()
    require_columns(data,["Representation","Horizon_Years",'Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95'],source)
    preferred=["NMR_score","Spearman-pruned NMR","Raw106 NMR"]; order=ordered_values(data,"Representation",preferred); y=np.arange(len(order))
    scale=1e5 if 'Delta_Uno_C'=="Delta_IBS" and np.nanmax(np.abs(data[['Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95']].to_numpy(float)))<1e-3 else 1.0
    fig,ax=plt.subplots(figsize=(9.2,5.4))
    for horizon,color,marker,offset in [(5.0,CNS["teal"],"o",-0.11),(10.0,CNS["blue"],"s",0.11)]:
        part=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),horizon)].copy(); part["Order"]=part["Representation"].map({x:i for i,x in enumerate(order)}); part=part.sort_values("Order")
        value=part['Delta_Uno_C'].to_numpy(float)*scale; lo=part['Delta_Uno_C_Lower95'].to_numpy(float)*scale; hi=part['Delta_Uno_C_Upper95'].to_numpy(float)*scale
        ax.errorbar(value,y+offset,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt=marker,color=color,ecolor=color,capsize=3,markersize=6.5,label=f"{horizon:g} years")
    ax.axvline(0,color=CNS["gray"],lw=1); ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlabel('Delta Uno IPCW C-index vs Clinical + PRS (95% CI)'+(" (×10⁻⁵)" if scale==1e5 else ""))
    ax.set_title('Figure 3B — Elastic-net sensitivity',loc="left",fontweight="bold"); clean_axis(ax,"x"); ax.legend(frameon=False); fig.tight_layout(); save_show(fig,'figure3B_elasticnet_sensitivity_delta_uno_plot_only')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3C — direct modeling-strategy comparison
try:
    source=find_unique('figure3C_elasticnet_minus_cox_delta_uno_cindex.csv'); data=pd.read_csv(source)
    require_columns(data,["Representation","Horizon_Years",'Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95'],source)
    preferred=["NMR_score","Spearman-pruned NMR","Raw106 NMR"]; order=ordered_values(data,"Representation",preferred); y=np.arange(len(order))
    scale=1e5 if 'Delta_Uno_C'=="Delta_IBS" and np.nanmax(np.abs(data[['Delta_Uno_C','Delta_Uno_C_Lower95','Delta_Uno_C_Upper95']].to_numpy(float)))<1e-3 else 1.0
    fig,ax=plt.subplots(figsize=(9.2,5.4))
    for horizon,color,marker,offset in [(5.0,CNS["teal"],"o",-0.11),(10.0,CNS["blue"],"s",0.11)]:
        part=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),horizon)].copy(); part["Order"]=part["Representation"].map({x:i for i,x in enumerate(order)}); part=part.sort_values("Order")
        value=part['Delta_Uno_C'].to_numpy(float)*scale; lo=part['Delta_Uno_C_Lower95'].to_numpy(float)*scale; hi=part['Delta_Uno_C_Upper95'].to_numpy(float)*scale
        ax.errorbar(value,y+offset,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt=marker,color=color,ecolor=color,capsize=3,markersize=6.5,label=f"{horizon:g} years")
    ax.axvline(0,color=CNS["gray"],lw=1,linestyle='--'); ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlabel('Delta Uno IPCW C-index: Elastic-net minus Cox PH (95% CI)'+(" (×10⁻⁵)" if scale==1e5 else ""))
    ax.set_title('Figure 3C — direct modeling-strategy comparison',loc="left",fontweight="bold"); clean_axis(ax,"x"); ax.legend(frameon=False); fig.tight_layout(); save_show(fig,'figure3C_elasticnet_minus_cox_delta_uno_plot_only')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3D — Cox primary prediction error
try:
    source=find_unique('figure3D_delta_ipcw_ibs.csv'); data=pd.read_csv(source)
    data=data.loc[data["Algorithm"].astype(str).eq('Cox PH')].copy()
    require_columns(data,["Representation","Horizon_Years",'Delta_IBS','Delta_IBS_Lower95','Delta_IBS_Upper95'],source)
    preferred=["NMR_score","Spearman-pruned NMR","Raw106 NMR"]; order=ordered_values(data,"Representation",preferred); y=np.arange(len(order))
    scale=1e5 if 'Delta_IBS'=="Delta_IBS" and np.nanmax(np.abs(data[['Delta_IBS','Delta_IBS_Lower95','Delta_IBS_Upper95']].to_numpy(float)))<1e-3 else 1.0
    fig,ax=plt.subplots(figsize=(9.2,5.4))
    for horizon,color,marker,offset in [(5.0,CNS["teal"],"o",-0.11),(10.0,CNS["blue"],"s",0.11)]:
        part=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),horizon)].copy(); part["Order"]=part["Representation"].map({x:i for i,x in enumerate(order)}); part=part.sort_values("Order")
        value=part['Delta_IBS'].to_numpy(float)*scale; lo=part['Delta_IBS_Lower95'].to_numpy(float)*scale; hi=part['Delta_IBS_Upper95'].to_numpy(float)*scale
        ax.errorbar(value,y+offset,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt=marker,color=color,ecolor=color,capsize=3,markersize=6.5,label=f"{horizon:g} years")
    ax.axvline(0,color=CNS["gray"],lw=1,linestyle='--'); ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlabel('Delta IPCW IBS vs Clinical + PRS; lower is better'+(" (×10⁻⁵)" if scale==1e5 else ""))
    ax.set_title('Figure 3D — Cox primary prediction error',loc="left",fontweight="bold"); clean_axis(ax,"x"); ax.legend(frameon=False); fig.tight_layout(); save_show(fig,'figure3D_cox_primary_delta_ibs_plot_only')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3D — Elastic-net sensitivity prediction error
try:
    source=find_unique('figure3D_delta_ipcw_ibs.csv'); data=pd.read_csv(source)
    data=data.loc[data["Algorithm"].astype(str).eq('Elastic-net Cox')].copy()
    require_columns(data,["Representation","Horizon_Years",'Delta_IBS','Delta_IBS_Lower95','Delta_IBS_Upper95'],source)
    preferred=["NMR_score","Spearman-pruned NMR","Raw106 NMR"]; order=ordered_values(data,"Representation",preferred); y=np.arange(len(order))
    scale=1e5 if 'Delta_IBS'=="Delta_IBS" and np.nanmax(np.abs(data[['Delta_IBS','Delta_IBS_Lower95','Delta_IBS_Upper95']].to_numpy(float)))<1e-3 else 1.0
    fig,ax=plt.subplots(figsize=(9.2,5.4))
    for horizon,color,marker,offset in [(5.0,CNS["teal"],"o",-0.11),(10.0,CNS["blue"],"s",0.11)]:
        part=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),horizon)].copy(); part["Order"]=part["Representation"].map({x:i for i,x in enumerate(order)}); part=part.sort_values("Order")
        value=part['Delta_IBS'].to_numpy(float)*scale; lo=part['Delta_IBS_Lower95'].to_numpy(float)*scale; hi=part['Delta_IBS_Upper95'].to_numpy(float)*scale
        ax.errorbar(value,y+offset,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt=marker,color=color,ecolor=color,capsize=3,markersize=6.5,label=f"{horizon:g} years")
    ax.axvline(0,color=CNS["gray"],lw=1,linestyle='--'); ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlabel('Delta IPCW IBS vs Clinical + PRS; lower is better'+(" (×10⁻⁵)" if scale==1e5 else ""))
    ax.set_title('Figure 3D — Elastic-net sensitivity prediction error',loc="left",fontweight="bold"); clean_axis(ax,"x"); ax.legend(frameon=False); fig.tight_layout(); save_show(fig,'figure3D_elasticnet_delta_ibs_plot_only')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=5, metric='continuous_nri', label_column='Representation',
        stem='figure3E_revised_cox_primary_5y_continuous_nri',
        filters={'Algorithm': 'Cox PH'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=5, metric='categorical_nri', label_column='Representation',
        stem='figure3E_revised_cox_primary_5y_categorical_nri',
        filters={'Algorithm': 'Cox PH'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=5, metric='idi', label_column='Representation',
        stem='figure3E_revised_cox_primary_5y_idi',
        filters={'Algorithm': 'Cox PH'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=10, metric='continuous_nri', label_column='Representation',
        stem='figure3E_revised_cox_primary_10y_continuous_nri',
        filters={'Algorithm': 'Cox PH'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=10, metric='categorical_nri', label_column='Representation',
        stem='figure3E_revised_cox_primary_10y_categorical_nri',
        filters={'Algorithm': 'Cox PH'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=10, metric='idi', label_column='Representation',
        stem='figure3E_revised_cox_primary_10y_idi',
        filters={'Algorithm': 'Cox PH'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=5, metric='continuous_nri', label_column='Representation',
        stem='figure3E_revised_elasticnet_sensitivity_5y_continuous_nri',
        filters={'Algorithm': 'Elastic-net Cox'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=5, metric='categorical_nri', label_column='Representation',
        stem='figure3E_revised_elasticnet_sensitivity_5y_categorical_nri',
        filters={'Algorithm': 'Elastic-net Cox'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=5, metric='idi', label_column='Representation',
        stem='figure3E_revised_elasticnet_sensitivity_5y_idi',
        filters={'Algorithm': 'Elastic-net Cox'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=10, metric='continuous_nri', label_column='Representation',
        stem='figure3E_revised_elasticnet_sensitivity_10y_continuous_nri',
        filters={'Algorithm': 'Elastic-net Cox'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=10, metric='categorical_nri', label_column='Representation',
        stem='figure3E_revised_elasticnet_sensitivity_10y_categorical_nri',
        filters={'Algorithm': 'Elastic-net Cox'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_reclassification_one(
        filename='figure3E_ipcw_nri_idi.csv',
        horizon=10, metric='idi', label_column='Representation',
        stem='figure3E_revised_elasticnet_sensitivity_10y_idi',
        filters={'Algorithm': 'Elastic-net Cox'},
    )
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3F — Cox PH, 5-year calibration
try:
    source=find_unique("figure3F_calibration.csv"); data=pd.read_csv(source)
    require_columns(data,["Algorithm","Representation","Horizon_Years","Predicted_Risk","Observed_KM_Risk"],source)
    data=data.loc[data["Algorithm"].astype(str).eq('Cox PH') & np.isclose(pd.to_numeric(data["Horizon_Years"]),5.0)].copy()
    colors={"Clinical + PRS":CNS["black"],"NMR_score":CNS["green"],"Spearman-pruned NMR":CNS["purple"],"Raw106 NMR":CNS["red"]}
    fig,ax=plt.subplots(figsize=(6.8,5.6)); maximum=float(np.nanmax(data[["Predicted_Risk","Observed_KM_Risk"]].to_numpy(float))); ax.plot([0,maximum],[0,maximum],"--",color=CNS["gray"],label="Ideal")
    for representation,part in data.groupby("Representation"):
        part=part.sort_values("Predicted_Risk"); ax.plot(part["Predicted_Risk"],part["Observed_KM_Risk"],marker="o",markersize=4,lw=1.8,color=colors.get(representation,CNS["gray"]),label=representation)
    ax.set_xlabel("Predicted 5-year CAD risk"); ax.set_ylabel("Observed 5-year CAD risk (KM)"); ax.set_title("Cox PH: 5-year calibration",loc="left",fontweight="bold")
    clean_axis(ax,"both"); ax.legend(frameon=False,fontsize=10); fig.tight_layout(); save_show(fig,"figure3F_cox_5y_calibration_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3F — Cox PH, 10-year calibration
try:
    source=find_unique("figure3F_calibration.csv"); data=pd.read_csv(source)
    require_columns(data,["Algorithm","Representation","Horizon_Years","Predicted_Risk","Observed_KM_Risk"],source)
    data=data.loc[data["Algorithm"].astype(str).eq('Cox PH') & np.isclose(pd.to_numeric(data["Horizon_Years"]),10.0)].copy()
    colors={"Clinical + PRS":CNS["black"],"NMR_score":CNS["green"],"Spearman-pruned NMR":CNS["purple"],"Raw106 NMR":CNS["red"]}
    fig,ax=plt.subplots(figsize=(6.8,5.6)); maximum=float(np.nanmax(data[["Predicted_Risk","Observed_KM_Risk"]].to_numpy(float))); ax.plot([0,maximum],[0,maximum],"--",color=CNS["gray"],label="Ideal")
    for representation,part in data.groupby("Representation"):
        part=part.sort_values("Predicted_Risk"); ax.plot(part["Predicted_Risk"],part["Observed_KM_Risk"],marker="o",markersize=4,lw=1.8,color=colors.get(representation,CNS["gray"]),label=representation)
    ax.set_xlabel("Predicted 10-year CAD risk"); ax.set_ylabel("Observed 10-year CAD risk (KM)"); ax.set_title("Cox PH: 10-year calibration",loc="left",fontweight="bold")
    clean_axis(ax,"both"); ax.legend(frameon=False,fontsize=10); fig.tight_layout(); save_show(fig,"figure3F_cox_10y_calibration_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3F — Elastic-net Cox, 5-year calibration
try:
    source=find_unique("figure3F_calibration.csv"); data=pd.read_csv(source)
    require_columns(data,["Algorithm","Representation","Horizon_Years","Predicted_Risk","Observed_KM_Risk"],source)
    data=data.loc[data["Algorithm"].astype(str).eq('Elastic-net Cox') & np.isclose(pd.to_numeric(data["Horizon_Years"]),5.0)].copy()
    colors={"Clinical + PRS":CNS["black"],"NMR_score":CNS["green"],"Spearman-pruned NMR":CNS["purple"],"Raw106 NMR":CNS["red"]}
    fig,ax=plt.subplots(figsize=(6.8,5.6)); maximum=float(np.nanmax(data[["Predicted_Risk","Observed_KM_Risk"]].to_numpy(float))); ax.plot([0,maximum],[0,maximum],"--",color=CNS["gray"],label="Ideal")
    for representation,part in data.groupby("Representation"):
        part=part.sort_values("Predicted_Risk"); ax.plot(part["Predicted_Risk"],part["Observed_KM_Risk"],marker="o",markersize=4,lw=1.8,color=colors.get(representation,CNS["gray"]),label=representation)
    ax.set_xlabel("Predicted 5-year CAD risk"); ax.set_ylabel("Observed 5-year CAD risk (KM)"); ax.set_title("Elastic-net Cox: 5-year calibration",loc="left",fontweight="bold")
    clean_axis(ax,"both"); ax.legend(frameon=False,fontsize=10); fig.tight_layout(); save_show(fig,"figure3F_elasticnet_5y_calibration_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3F — Elastic-net Cox, 10-year calibration
try:
    source=find_unique("figure3F_calibration.csv"); data=pd.read_csv(source)
    require_columns(data,["Algorithm","Representation","Horizon_Years","Predicted_Risk","Observed_KM_Risk"],source)
    data=data.loc[data["Algorithm"].astype(str).eq('Elastic-net Cox') & np.isclose(pd.to_numeric(data["Horizon_Years"]),10.0)].copy()
    colors={"Clinical + PRS":CNS["black"],"NMR_score":CNS["green"],"Spearman-pruned NMR":CNS["purple"],"Raw106 NMR":CNS["red"]}
    fig,ax=plt.subplots(figsize=(6.8,5.6)); maximum=float(np.nanmax(data[["Predicted_Risk","Observed_KM_Risk"]].to_numpy(float))); ax.plot([0,maximum],[0,maximum],"--",color=CNS["gray"],label="Ideal")
    for representation,part in data.groupby("Representation"):
        part=part.sort_values("Predicted_Risk"); ax.plot(part["Predicted_Risk"],part["Observed_KM_Risk"],marker="o",markersize=4,lw=1.8,color=colors.get(representation,CNS["gray"]),label=representation)
    ax.set_xlabel("Predicted 10-year CAD risk"); ax.set_ylabel("Observed 10-year CAD risk (KM)"); ax.set_title("Elastic-net Cox: 10-year calibration",loc="left",fontweight="bold")
    clean_axis(ax,"both"); ax.legend(frameon=False,fontsize=10); fig.tight_layout(); save_show(fig,"figure3F_elasticnet_10y_calibration_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3G — Cox PH, 5-year survival DCA
try:
    source=find_unique("figure3G_survival_dca.csv"); data=pd.read_csv(source)
    require_columns(data,["Algorithm","Horizon_Years","Strategy","Threshold","Net_Benefit"],source)
    data=data.loc[data["Algorithm"].astype(str).eq('Cox PH') & np.isclose(pd.to_numeric(data["Horizon_Years"]),5.0)].copy()
    colors={"Clinical + PRS":CNS["black"],"NMR_score":CNS["green"],"Spearman-pruned NMR":CNS["purple"],"Raw106 NMR":CNS["red"],"Treat all":CNS["gray"],"Treat none":CNS["gray"]}
    fig,ax=plt.subplots(figsize=(7.8,5.7))
    for strategy,part in data.groupby("Strategy",sort=False):
        part=part.sort_values("Threshold"); ax.plot(part["Threshold"].astype(float)*100,part["Net_Benefit"],color=colors.get(strategy,CNS["gray"]),lw=2 if strategy=="NMR_score" else 1.5,ls="--" if strategy=="Treat all" else (":" if strategy=="Treat none" else "-"),label=strategy)
    ax.set_xlabel("5-year CAD risk threshold (%)"); ax.set_ylabel("Net benefit"); ax.set_title("Cox PH: 5-year survival DCA",loc="left",fontweight="bold")
    clean_axis(ax,"both"); ax.legend(frameon=False,fontsize=10); fig.tight_layout(); save_show(fig,"figure3G_cox_5y_dca_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3G — Cox PH, 10-year survival DCA
try:
    source=find_unique("figure3G_survival_dca.csv"); data=pd.read_csv(source)
    require_columns(data,["Algorithm","Horizon_Years","Strategy","Threshold","Net_Benefit"],source)
    data=data.loc[data["Algorithm"].astype(str).eq('Cox PH') & np.isclose(pd.to_numeric(data["Horizon_Years"]),10.0)].copy()
    colors={"Clinical + PRS":CNS["black"],"NMR_score":CNS["green"],"Spearman-pruned NMR":CNS["purple"],"Raw106 NMR":CNS["red"],"Treat all":CNS["gray"],"Treat none":CNS["gray"]}
    fig,ax=plt.subplots(figsize=(7.8,5.7))
    for strategy,part in data.groupby("Strategy",sort=False):
        part=part.sort_values("Threshold"); ax.plot(part["Threshold"].astype(float)*100,part["Net_Benefit"],color=colors.get(strategy,CNS["gray"]),lw=2 if strategy=="NMR_score" else 1.5,ls="--" if strategy=="Treat all" else (":" if strategy=="Treat none" else "-"),label=strategy)
    ax.set_xlabel("10-year CAD risk threshold (%)"); ax.set_ylabel("Net benefit"); ax.set_title("Cox PH: 10-year survival DCA",loc="left",fontweight="bold")
    clean_axis(ax,"both"); ax.legend(frameon=False,fontsize=10); fig.tight_layout(); save_show(fig,"figure3G_cox_10y_dca_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3G — Elastic-net Cox, 5-year survival DCA
try:
    source=find_unique("figure3G_survival_dca.csv"); data=pd.read_csv(source)
    require_columns(data,["Algorithm","Horizon_Years","Strategy","Threshold","Net_Benefit"],source)
    data=data.loc[data["Algorithm"].astype(str).eq('Elastic-net Cox') & np.isclose(pd.to_numeric(data["Horizon_Years"]),5.0)].copy()
    colors={"Clinical + PRS":CNS["black"],"NMR_score":CNS["green"],"Spearman-pruned NMR":CNS["purple"],"Raw106 NMR":CNS["red"],"Treat all":CNS["gray"],"Treat none":CNS["gray"]}
    fig,ax=plt.subplots(figsize=(7.8,5.7))
    for strategy,part in data.groupby("Strategy",sort=False):
        part=part.sort_values("Threshold"); ax.plot(part["Threshold"].astype(float)*100,part["Net_Benefit"],color=colors.get(strategy,CNS["gray"]),lw=2 if strategy=="NMR_score" else 1.5,ls="--" if strategy=="Treat all" else (":" if strategy=="Treat none" else "-"),label=strategy)
    ax.set_xlabel("5-year CAD risk threshold (%)"); ax.set_ylabel("Net benefit"); ax.set_title("Elastic-net Cox: 5-year survival DCA",loc="left",fontweight="bold")
    clean_axis(ax,"both"); ax.legend(frameon=False,fontsize=10); fig.tight_layout(); save_show(fig,"figure3G_elasticnet_5y_dca_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 3G — Elastic-net Cox, 10-year survival DCA
try:
    source=find_unique("figure3G_survival_dca.csv"); data=pd.read_csv(source)
    require_columns(data,["Algorithm","Horizon_Years","Strategy","Threshold","Net_Benefit"],source)
    data=data.loc[data["Algorithm"].astype(str).eq('Elastic-net Cox') & np.isclose(pd.to_numeric(data["Horizon_Years"]),10.0)].copy()
    colors={"Clinical + PRS":CNS["black"],"NMR_score":CNS["green"],"Spearman-pruned NMR":CNS["purple"],"Raw106 NMR":CNS["red"],"Treat all":CNS["gray"],"Treat none":CNS["gray"]}
    fig,ax=plt.subplots(figsize=(7.8,5.7))
    for strategy,part in data.groupby("Strategy",sort=False):
        part=part.sort_values("Threshold"); ax.plot(part["Threshold"].astype(float)*100,part["Net_Benefit"],color=colors.get(strategy,CNS["gray"]),lw=2 if strategy=="NMR_score" else 1.5,ls="--" if strategy=="Treat all" else (":" if strategy=="Treat none" else "-"),label=strategy)
    ax.set_xlabel("10-year CAD risk threshold (%)"); ax.set_ylabel("Net benefit"); ax.set_title("Elastic-net Cox: 10-year survival DCA",loc="left",fontweight="bold")
    clean_axis(ax,"both"); ax.legend(frameon=False,fontsize=10); fig.tight_layout(); save_show(fig,"figure3G_elasticnet_10y_dca_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    source=find_unique("supplement_elasticnet_minus_cox_delta_harrell_cindex.csv"); data=pd.read_csv(source)
    require_columns(data,["Representation","Delta_Harrell_C","Delta_Harrell_C_Lower95","Delta_Harrell_C_Upper95"],source)
    data=data.reset_index(drop=True); y=np.arange(len(data)); value=data["Delta_Harrell_C"].to_numpy(float); lo=data["Delta_Harrell_C_Lower95"].to_numpy(float); hi=data["Delta_Harrell_C_Upper95"].to_numpy(float)
    fig,ax=plt.subplots(figsize=(8.5,4.8)); ax.errorbar(value,y,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt="o",color=CNS["purple"],ecolor=CNS["purple"],capsize=3)
    ax.axvline(0,color=CNS["gray"],lw=1); ax.set_yticks(y); ax.set_yticklabels(data["Representation"]); ax.invert_yaxis(); ax.set_xlabel("Delta Harrell C-index: Elastic-net minus Cox PH (95% CI)")
    ax.set_title("Harrell C sensitivity",loc="left",fontweight="bold"); clean_axis(ax,"x"); fig.tight_layout(); save_show(fig,"supplement_result3_delta_harrell_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, FormatStrFormatter, MultipleLocator

summary_source = find_unique(
    "result4_all_models_uno_ipcw_summary.csv"
)

delta_source = find_unique(
    "result4_all_models_delta_uno_ipcw_vs_cox.csv"
)

summary = pd.read_csv(
    summary_source
)

delta = pd.read_csv(
    delta_source
)

require_columns(
    summary,
    [
        "Representation",
        "Model",
        "Uno_C",
        "Uno_C_Lower95",
        "Uno_C_Upper95",
    ],
    summary_source,
)

require_columns(
    delta,
    [
        "Representation",
        "Model",
        "Delta_Uno_C_vs_Cox",
        "Delta_Uno_C_Lower95",
        "Delta_Uno_C_Upper95",
    ],
    delta_source,
)

summary["Model"] = summary["Model"].replace({
    "XGBoost Cox": "XGBoost survival",
})

delta["Model"] = delta["Model"].replace({
    "XGBoost Cox": "XGBoost survival",
})

for column in [
    "Uno_C",
    "Uno_C_Lower95",
    "Uno_C_Upper95",
]:
    summary[column] = pd.to_numeric(
        summary[column],
        errors="coerce",
    )

for column in [
    "Delta_Uno_C_vs_Cox",
    "Delta_Uno_C_Lower95",
    "Delta_Uno_C_Upper95",
]:
    delta[column] = pd.to_numeric(
        delta[column],
        errors="coerce",
    )

representation_order = [
    "NMR_score",
    "Spearman-pruned NMR",
    "Raw106 NMR",
]

representation_labels = {
    "NMR_score": "NMR score",
    "Spearman-pruned NMR": "Spearman-pruned",
    "Raw106 NMR": "Raw106",
}

model_order = [
    "Cox PH",
    "Elastic-net Cox",
    "XGBoost survival",
    "DeepSurv",
]

comparison_models = [
    "Elastic-net Cox",
    "XGBoost survival",
    "DeepSurv",
]

model_colors = {
    "Cox PH": "#333333",
    "Elastic-net Cox": CNS["blue"],
    "XGBoost survival": CNS["green"],
    "DeepSurv": CNS["red"],
}

model_markers = {
    "Cox PH": "o",
    "Elastic-net Cox": "s",
    "XGBoost survival": "^",
    "DeepSurv": "D",
}

summary_offsets = {
    "Cox PH": -0.21,
    "Elastic-net Cox": -0.07,
    "XGBoost survival": 0.07,
    "DeepSurv": 0.21,
}

delta_offsets = {
    "Elastic-net Cox": -0.14,
    "XGBoost survival": 0.00,
    "DeepSurv": 0.14,
}

y_base = np.arange(
    len(representation_order)
)

fig, (
    ax_absolute,
    ax_delta,
) = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13.0, 5.4),
    sharey=True,
    gridspec_kw={
        "width_ratios": [0.88, 1.00],
        "wspace": 0.13,
    },
)

row_backgrounds = [
    "#F4F0F3",
    "#FBF8F4",
    "#F4F0F3",
]

for position, background in zip(
    y_base,
    row_backgrounds,
):
    for ax in [
        ax_absolute,
        ax_delta,
    ]:
        ax.axhspan(
            position - 0.43,
            position + 0.43,
            color=background,
            linewidth=0,
            zorder=0,
        )

for model in model_order:
    model_data = summary.loc[
        summary["Model"]
        .astype(str)
        .eq(model)
    ].copy()

    model_data = (
        model_data
        .set_index("Representation")
        .reindex(representation_order)
        .reset_index()
    )

    estimate = model_data[
        "Uno_C"
    ].to_numpy(float)

    lower = model_data[
        "Uno_C_Lower95"
    ].to_numpy(float)

    upper = model_data[
        "Uno_C_Upper95"
    ].to_numpy(float)

    positions = (
        y_base
        + summary_offsets[model]
    )

    marker_face = (
        "white"
        if model == "Elastic-net Cox"
        else model_colors[model]
    )

    ax_absolute.errorbar(
        estimate,
        positions,
        xerr=[
            np.maximum(
                0,
                estimate - lower,
            ),
            np.maximum(
                0,
                upper - estimate,
            ),
        ],
        fmt=model_markers[model],
        color=model_colors[model],
        ecolor=model_colors[model],
        markerfacecolor=marker_face,
        markeredgecolor=model_colors[model],
        markeredgewidth=1.3,
        markersize=7,
        elinewidth=1.35,
        capsize=3,
        capthick=1.1,
        linestyle="none",
        zorder=3,
    )

absolute_lower = float(
    summary["Uno_C_Lower95"].min()
)

absolute_upper = float(
    summary["Uno_C_Upper95"].max()
)

absolute_span = (
    absolute_upper
    - absolute_lower
)

absolute_padding = max(
    absolute_span * 0.12,
    0.0008,
)

import math

lower = absolute_lower - absolute_padding
upper = absolute_upper + absolute_padding
new_lower = math.floor(lower / 0.01) * 0.01
new_upper = math.ceil(upper / 0.01) * 0.01

ax_absolute.set_xlim(new_lower, new_upper)

ax_absolute.xaxis.set_major_locator(
    MultipleLocator(0.01)
)

ax_absolute.xaxis.set_major_formatter(
    FormatStrFormatter("%.2f")
)

delta_scale = 1000.0

for model in comparison_models:
    model_data = delta.loc[
        delta["Model"]
        .astype(str)
        .eq(model)
    ].copy()

    model_data = (
        model_data
        .set_index("Representation")
        .reindex(representation_order)
        .reset_index()
    )

    estimate = (
        model_data[
            "Delta_Uno_C_vs_Cox"
        ].to_numpy(float)
        * delta_scale
    )

    lower = (
        model_data[
            "Delta_Uno_C_Lower95"
        ].to_numpy(float)
        * delta_scale
    )

    upper = (
        model_data[
            "Delta_Uno_C_Upper95"
        ].to_numpy(float)
        * delta_scale
    )

    positions = (
        y_base
        + delta_offsets[model]
    )

    marker_face = (
        "white"
        if model == "Elastic-net Cox"
        else model_colors[model]
    )

    ax_delta.errorbar(
        estimate,
        positions,
        xerr=[
            np.maximum(
                0,
                estimate - lower,
            ),
            np.maximum(
                0,
                upper - estimate,
            ),
        ],
        fmt=model_markers[model],
        color=model_colors[model],
        ecolor=model_colors[model],
        markerfacecolor=marker_face,
        markeredgecolor=model_colors[model],
        markeredgewidth=1.3,
        markersize=7,
        elinewidth=1.35,
        capsize=3,
        capthick=1.1,
        linestyle="none",
        zorder=3,
    )

ax_delta.axvline(
    0,
    color=CNS["gray"],
    linestyle="--",
    linewidth=1.0,
    zorder=1,
)

scaled_lower = (
    delta["Delta_Uno_C_Lower95"]
    .to_numpy(float)
    * delta_scale
)

scaled_upper = (
    delta["Delta_Uno_C_Upper95"]
    .to_numpy(float)
    * delta_scale
)

delta_limit = max(
    abs(float(np.nanmin(scaled_lower))),
    abs(float(np.nanmax(scaled_upper))),
)

delta_padding = max(
    delta_limit * 0.12,
    0.20,
)

ax_delta.set_xlim(
    -delta_limit - delta_padding,
    delta_limit + delta_padding,
)

ax_delta.set_title(
    "Change relative to Cox PH",
    fontsize=13,
    fontweight="bold",
    pad=10,
)

ax_delta.set_xlabel(
    "Difference in Uno C-index (×10⁻³)",
    labelpad=13,
)

ax_delta.xaxis.set_major_locator(
    MaxNLocator(nbins=5)
)

ax_absolute.set_yticks(
    y_base
)

ax_absolute.set_yticklabels([
    representation_labels[value]
    for value in representation_order
])

ax_absolute.set_ylim(
    len(representation_order) - 0.52,
    -0.52,
)

ax_absolute.tick_params(
    axis="y",
    length=0,
    pad=12,
    labelsize=12,
)

ax_delta.tick_params(
    axis="y",
    left=False,
    labelleft=False,
)

for ax in [
    ax_absolute,
    ax_delta,
]:

    ax.tick_params(
        axis="x",
        top=True,
        labeltop=True,
        bottom=False,
        labelbottom=False,
        length=4,
    )

    ax.grid(
        axis="x",
        color="#D5D5D5",
        linewidth=0.7,
        alpha=0.8,
    )

    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#555555")
        spine.set_linewidth(0.8)

legend_handles = []

for model in model_order:
    marker_face = (
        "white"
        if model == "Elastic-net Cox"
        else model_colors[model]
    )

    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker=model_markers[model],
            linestyle="none",
            markersize=7,
            markerfacecolor=marker_face,
            markeredgecolor=model_colors[model],
            markeredgewidth=1.3,
            color=model_colors[model],
            label=model,
        )
    )

fig.legend(
    handles=legend_handles,
    title="Survival algorithm",
    frameon=False,
    ncol=4,
    loc="lower center",
    bbox_to_anchor=(0.58, 0.01),
    fontsize=10.5,
    title_fontsize=11,
    columnspacing=1.7,
    handletextpad=0.5,
)

fig.suptitle(
    "Predictive performance across NMR representations and algorithms",
    x=0.075,
    y=0.98,
    ha="left",
    fontsize=15,
    fontweight="bold",
)

fig.subplots_adjust(
    left=0.18,
    right=0.985,
    top=0.78,
    bottom=0.23,
)

save_show(
    fig,
    "figure4_combined_absolute_and_delta_uno",
)


In [ ]:
try:
    source=find_unique("result4_all_models_delta_uno_ipcw_vs_cox.csv"); data=pd.read_csv(source)
    require_columns(data,["Representation","Model","Delta_Uno_C_vs_Cox","Delta_Uno_C_Lower95","Delta_Uno_C_Upper95"],source)
    representations=ordered_values(data,"Representation",["NMR_score","Spearman-pruned NMR","Raw106 NMR"]); models=ordered_values(data,"Model",["Elastic-net Cox","XGBoost survival","DeepSurv"])
    colors={"Elastic-net Cox":CNS["blue"],"XGBoost survival":CNS["green"],"DeepSurv":CNS["red"]}; markers={"Elastic-net Cox":"s","XGBoost survival":"^","DeepSurv":"D"}
    offsets=dict(zip(models,np.linspace(-0.22,0.22,len(models)))); y=np.arange(len(representations)); fig,ax=plt.subplots(figsize=(9.4,5.3))
    for model in models:
        part=data.loc[data["Model"].astype(str).eq(model)].copy(); part["Order"]=part["Representation"].map({x:i for i,x in enumerate(representations)}); part=part.sort_values("Order")
        value=part["Delta_Uno_C_vs_Cox"].to_numpy(float); lo=part["Delta_Uno_C_Lower95"].to_numpy(float); hi=part["Delta_Uno_C_Upper95"].to_numpy(float)
        ax.errorbar(value,y+offsets[model],xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt=markers.get(model,"o"),color=colors.get(model,CNS["gray"]),ecolor=colors.get(model,CNS["gray"]),capsize=3,label=model)
    ax.axvline(0,color=CNS["gray"],ls="--",lw=1); ax.set_yticks(y); ax.set_yticklabels(representations); ax.invert_yaxis(); ax.set_xlabel("Difference in 10-year Uno IPCW C-index relative to Cox PH (95% CI)")
    ax.set_title("Difference in discrimination relative to Cox",loc="left",fontweight="bold"); clean_axis(ax,"x"); ax.legend(frameon=False,ncol=1, loc="center left",bbox_to_anchor=(1.02, 0.5))
    fig.tight_layout(); save_show(fig,"figure4B_delta_uno_vs_cox_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_calibration(10.0, 'figure5A_revised_10y_calibration_CAD')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_observed_predicted(10.0, 'figure5B_revised_10y_observed_vs_predicted_CAD')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Figure 5C — 10-year clinical utility

try:
    horizon = 10.0
    threshold_min = 3.0
    threshold_max = 15.0

    reference_model = "Clinical_plus_PRS"
    joint_model = "Clinical_plus_PRS_plus_NMR_score"

    dca = load_dca(horizon).copy()

    dca["Threshold"] = pd.to_numeric(
        dca["Threshold"], errors="coerce"
    )
    dca["Net_Benefit"] = pd.to_numeric(
        dca["Net_Benefit"], errors="coerce"
    )
    dca["Threshold_pct"] = dca["Threshold"] * 100

    dca = dca.loc[
        dca["Threshold_pct"].between(
            threshold_min, threshold_max
        )
    ].copy()

    if dca.empty:
        raise RuntimeError(
            "No 10-year DCA data at thresholds from 3% to 15%."
        )

    model_data = dca.loc[dca["Feature_Set"].notna()].copy()

    model_pivot = model_data.pivot_table(
        index="Threshold",
        columns="Feature_Set",
        values="Net_Benefit",
        aggfunc="mean",
    ).sort_index()

    required_models = [reference_model, joint_model]
    missing_models = [
        model for model in required_models
        if model not in model_pivot.columns
    ]

    if missing_models:
        raise RuntimeError(
            f"Missing DCA models: {missing_models}; "
            f"Available models={list(model_pivot.columns)}"
        )

    model_pivot = model_pivot[
        [reference_model, joint_model]
    ].dropna()

    threshold_pct = model_pivot.index.to_numpy(float) * 100

    reference_nb = (
        model_pivot[reference_model].to_numpy(float)
    )
    joint_nb = (
        model_pivot[joint_model].to_numpy(float)
    )

    strategy_lower = (
        dca["Strategy"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    treat_all = dca.loc[
        strategy_lower.eq("treat all")
    ].sort_values("Threshold")

    source_output = pd.DataFrame({
        "Threshold": model_pivot.index.to_numpy(float),
        "Threshold_Percent": threshold_pct,
        "Clinical_plus_PRS_Net_Benefit": reference_nb,
        "Joint_Model_Net_Benefit": joint_nb,
        "Delta_Net_Benefit": joint_nb - reference_nb,
        "Delta_Net_Benefit_Per1000":
            (joint_nb - reference_nb) * 1000,
    })

    source_output.to_csv(
        TABLE_DIR / "figure5C_10y_core_model_clinical_utility.csv",
        index=False,
    )

    fig, (ax_a, ax_b) = plt.subplots(
        2,
        1,
        figsize=(8.5, 8.0),
        sharex=True,
        gridspec_kw={
            "height_ratios": [1.45, 1.0],
            "hspace": 0.13,
        },
    )

    # Treat-all
    if not treat_all.empty:
        ax_a.plot(
            treat_all["Threshold"] * 100,
            treat_all["Net_Benefit"] * 100,
            color="#A0A0A0",
            linestyle=(0, (5, 3)),
            linewidth=1.35,
            label="Treat all",
            zorder=1,
        )

    # Treat-none
    ax_a.axhline(
        0,
        color="#777777",
        linestyle=(0, (1.5, 2.2)),
        linewidth=1.25,
        label="Treat none",
        zorder=1,
    )

    
    mark_every = 1

    # Clinical + PRS
    ax_a.plot(
        threshold_pct,
        reference_nb * 100,
        color=CNS["blue"],
        linestyle=(0, (5, 2)),
        linewidth=2.0,
        marker="o",
        markevery=mark_every,
        markersize=5.0,
        markerfacecolor="white",
        markeredgecolor=CNS["blue"],
        markeredgewidth=1.1,
        label="Clinical + PRS",
        zorder=3,
    )

    # Clinical + PRS + NMR_score
    ax_a.plot(
        threshold_pct,
        joint_nb * 100,
        color=CNS["red"],
        linestyle="-",
        linewidth=2.8,
        marker="D",
        markevery=mark_every,
        markersize=5.0,
        markerfacecolor="white",
        markeredgecolor=CNS["red"],
        markeredgewidth=1.2,
        label="Clinical + PRS + NMR score",
        zorder=4,
    )

    model_values_per100 = np.concatenate([
        reference_nb * 100,
        joint_nb * 100,
        np.array([0.0]),
    ])

    model_min = float(np.nanmin(model_values_per100))
    model_max = float(np.nanmax(model_values_per100))
    model_span = max(model_max - model_min, 0.5)

    y_lower = min(-0.05, model_min - 0.02 * model_span)
    y_upper = model_max + 0.05 * model_span

    ax_a.set_ylim(y_lower, y_upper)
    ax_a.set_ylabel("Net benefit\n(per 100 individuals)")

    ax_a.text(
        -0.09,
        1.03,
        "",
        transform=ax_a.transAxes,
        fontsize=16,
        fontweight="bold",
        va="bottom",
    )

    ax_a.set_title(
        "Clinical utility of adding NMR score",
        loc="left",
        fontsize=15,
        fontweight="bold",
        pad=12,
    )

    ax_a.legend(
        frameon=False,
        loc="upper right",
        ncol=2,
        fontsize=10.2,
        columnspacing=1.4,
        handlelength=3.0,
        handletextpad=0.7,
    )

    ax_a.grid(
        axis="y",
        color=CNS["light_gray"],
        linewidth=0.7,
        alpha=0.65,
    )
    ax_a.grid(axis="x", visible=False)
    ax_a.set_axisbelow(True)
    ax_a.spines["top"].set_visible(False)
    ax_a.spines["right"].set_visible(False)

    delta_per1000 = (
        joint_nb - reference_nb
    ) * 1000

    ax_b.axhline(
        0,
        color="#707070",
        linestyle=(0, (1.5, 2.2)),
        linewidth=1.2,
        zorder=1,
    )

    ax_b.fill_between(
        threshold_pct,
        0,
        delta_per1000,
        where=(delta_per1000 >= 0),
        interpolate=True,
        color=CNS["red"],
        alpha=0.14,
        linewidth=0,
        zorder=1,
    )

    ax_b.fill_between(
        threshold_pct,
        0,
        delta_per1000,
        where=(delta_per1000 < 0),
        interpolate=True,
        color=CNS["gray"],
        alpha=0.13,
        linewidth=0,
        zorder=1,
    )

    ax_b.plot(
        threshold_pct,
        delta_per1000,
        color=CNS["red"],
        linewidth=2.5,
        marker="o",
        markevery=mark_every,
        markersize=4.8,
        markerfacecolor="white",
        markeredgecolor=CNS["red"],
        markeredgewidth=1.1,
        zorder=3,
    )

    ax_b.text(
        -0.09,
        1.03,
        "",
        transform=ax_b.transAxes,
        fontsize=16,
        fontweight="bold",
        va="bottom",
    )

    ax_b.text(
        0.98,
        0.95,
        "Joint model − Clinical + PRS",
        transform=ax_b.transAxes,
        ha="right",
        va="top",
        fontsize=10.5,
        color=CNS["red"],
        fontweight="bold",
        zorder=100,
    )

    ax_b.set_xlim(threshold_min, threshold_max)
    ax_b.set_xticks(np.arange(4, 16, 2))

    ax_b.set_xlabel("10-year CAD risk threshold (%)")
    ax_b.set_ylabel(
        "Incremental net benefit\n(per 1,000 individuals)"
    )

    ax_b.grid(
        axis="y",
        color=CNS["light_gray"],
        linewidth=0.7,
        alpha=0.65,
    )
    ax_b.grid(axis="x", visible=False)
    ax_b.set_axisbelow(True)
    ax_b.spines["top"].set_visible(False)
    ax_b.spines["right"].set_visible(False)

    fig.text(
        0.125,
        0.012,
        "The upper panel is scaled around the model curves; "
        "treat-all values outside the displayed range are clipped.",
        ha="left",
        va="bottom",
        fontsize=9.3,
        color=CNS["gray"],
    )

    fig.subplots_adjust(
        left=0.15,
        right=0.97,
        top=0.94,
        bottom=0.12,
    )

    save_show(
        fig,
        "figure5C_10y_core_model_clinical_utility",
    )

except FileNotFoundError as error:
    print("SKIP — missing DCA source file: ", error)


In [ ]:
try:

    plot_incremental_dca(10.0, 'figure5D_revised_joint_vs_clinical_PRS_delta_NB_3to15pct', (3.0, 15.0))
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_calibration(5.0, 'supplementary_figure5A_revised_5y_calibration_CAD')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_observed_predicted(5.0, 'supplementary_figure5B_revised_5y_observed_vs_predicted_CAD')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_standard_dca(5.0, 'supplementary_figure5C_revised_5y_survival_DCA_CAD', (1.0, 8.0))
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    plot_incremental_dca(5.0, 'supplementary_figure5D_revised_joint_vs_clinical_PRS_delta_NB', (1.0, 8.0))
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    source = find_unique('result5_final_joint_prs_nmr_observed_predicted_risk.csv')
    data = pd.read_csv(source); require_columns(data, ['Horizon_Years','PRS_Tertile','NMR_Tertile','Observed_KM_Risk','N'], source)
    use = data.loc[np.isclose(pd.to_numeric(data['Horizon_Years']), 10.0)].copy()
    matrix = np.full((3,3), np.nan); n_matrix = np.full((3,3), np.nan)
    for _, row in use.iterrows():
        i = PRS_ORDER.index(row['PRS_Tertile']); j = NMR_ORDER.index(row['NMR_Tertile'])
        matrix[i,j] = float(row['Observed_KM_Risk'])*100; n_matrix[i,j] = float(row['N'])
    fig, ax = plt.subplots(figsize=(7.1, 5.9))
    cmap = LinearSegmentedColormap.from_list('risk_cmap', ['#EDF5FB','#F39B7F','#B2182B'])
    image = ax.imshow(matrix, cmap=cmap, aspect='auto')
    for i in range(3):
        for j in range(3): ax.text(j, i, f'{matrix[i,j]:.1f}%\nn={int(n_matrix[i,j]):,}', ha='center', va='center', fontsize=11)
    ax.set_xticks(range(3)); ax.set_xticklabels([NMR_LABELS[x] for x in NMR_ORDER], rotation=20, ha='right')
    ax.set_yticks(range(3)); ax.set_yticklabels([PRS_LABELS[x] for x in PRS_ORDER])
    ax.set_xlabel('NMR_score tertile'); ax.set_ylabel('CAD PRS tertile')
    ax.set_title('Joint 10-year CAD risk', loc='left', fontweight='bold'); ax.tick_params(length=0)
    for spine in ax.spines.values(): spine.set_visible(False)
    cb = plt.colorbar(image, ax=ax, fraction=0.046, pad=0.02); cb.set_label('Observed 10-year CAD risk, 1-KM (%)')
    use.to_csv(TABLE_DIR / 'figure6A_revised_joint_10y_CAD_risk.csv', index=False)
    fig.tight_layout(); save_show(fig, 'figure6A_revised_joint_10y_CAD_risk_heatmap')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    source = find_unique('result5_final_joint_prs_nmr_km_curve.csv')
    data = pd.read_csv(source); require_columns(data, ['Group_Clean','Time','Cumulative_CHD_Risk'], source)
    logrank_source = find_unique('supplementary_table_result5_final_joint_logrank_test.csv')
    logrank = pd.read_csv(logrank_source); p_value = float(logrank['P_Value'].iloc[0])
    fig, ax = plt.subplots(figsize=(12.6, 7.2))
    for group in GROUP_ORDER:
        use = data.loc[data['Group_Clean'].astype(str).eq(group)].sort_values('Time')
        ax.step(use['Time'], use['Cumulative_CHD_Risk'].astype(float)*100, where='post', color=KM_COLORS[group],
                lw=2.0 if group in {'PRS_low__NMR_low','PRS_high__NMR_high'} else 1.35, label=GROUP_LABELS[group])
    ax.set_xlim(0,10); ax.set_ylim(bottom=0); ax.set_xlabel('Follow-up time (years)')
    ax.set_ylabel('Kaplan-Meier cumulative CAD risk (%)')
    ax.set_title('Cumulative CAD risk across joint risk strata', loc='left', fontweight='bold')
    clean_axis(ax, 'both'); ax.legend(title='PRS tertile / NMR_score tertile', frameon=False, fontsize=10.5,
            title_fontsize=11, loc='upper left', bbox_to_anchor=(1.02,1.0), borderaxespad=0)
    p_label = 'Log-rank P < 0.0001' if p_value < 1e-4 else f'Log-rank P = {p_value:.3g}'
    ax.text(0.98,0.03,p_label,transform=ax.transAxes,ha='right',va='bottom',fontsize=10.5)
    data.rename(columns={'Cumulative_CHD_Risk':'Cumulative_CAD_Risk'}).to_csv(TABLE_DIR / 'figure6B_revised_joint_KM_CAD.csv', index=False)
    fig.tight_layout(); save_show(fig, 'figure6B_revised_joint_KM_cumulative_CAD_risk')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    from matplotlib.ticker import NullLocator

    source = find_unique(
        "result5_final_figure6_adjusted_hr_source.csv"
    )

    data = pd.read_csv(source)

    require_columns(
        data,
        [
            "Group_Clean",
            "Group_Label",
            "HR",
            "HR_CI_Lower",
            "HR_CI_Upper",
        ],
        source,
    )

    for column in [
        "HR",
        "HR_CI_Lower",
        "HR_CI_Upper",
    ]:
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce",
        )

    data = data.dropna(
        subset=[
            "Group_Clean",
            "Group_Label",
            "HR",
            "HR_CI_Lower",
            "HR_CI_Upper",
        ]
    ).copy()

    data["Group_Clean"] = pd.Categorical(
        data["Group_Clean"],
        categories=GROUP_ORDER,
        ordered=True,
    )

    data = (
        data.sort_values("Group_Clean")
        .reset_index(drop=True)
    )

    data["Display_Label"] = (
        data["Group_Label"]
        .astype(str)
        .str.replace(
            "NMR_score",
            "NMR score",
            regex=False,
        )
    )

    y = np.arange(len(data))

    estimate = data["HR"].to_numpy(float)
    lower = data["HR_CI_Lower"].to_numpy(float)
    upper = data["HR_CI_Upper"].to_numpy(float)

    if (
        np.any(lower > estimate)
        or np.any(upper < estimate)
        or np.any(lower <= 0)
    ):
        raise RuntimeError(
            "Figure 6C contains an invalid HR or 95% CI."
        )

    data.to_csv(
        TABLE_DIR
        / "figure6C_revised_adjusted_HR_CAD.csv",
        index=False,
    )

    fig, ax = plt.subplots(
        figsize=(9.6, 7.0)
    )

    for position in y:
        if position % 2 == 0:
            ax.axhspan(
                position - 0.46,
                position + 0.46,
                color="#F6F6F6",
                linewidth=0,
                zorder=0,
            )

    ax.errorbar(
        estimate,
        y,
        xerr=np.vstack([
            np.maximum(
                0,
                estimate - lower,
            ),
            np.maximum(
                0,
                upper - estimate,
            ),
        ]),
        fmt="o",
        color=CNS["blue"],
        ecolor=CNS["blue"],
        markerfacecolor=CNS["blue"],
        markeredgecolor="white",
        markeredgewidth=0.7,
        markersize=6.5,
        elinewidth=1.25,
        capsize=3,
        capthick=1.0,
        linestyle="none",
        zorder=3,
    )

    ax.axvline(
        1.0,
        color=CNS["gray"],
        linestyle=(0, (3, 2)),
        linewidth=1.0,
        zorder=1,
    )

    ax.set_xscale(
        "log",
        base=2,
    )

    ax.set_xticks([
        1.0,
        2.0,
        4.0,
    ])

    ax.set_xticklabels([
        "1",
        "2",
        "4",
    ])

    ax.xaxis.set_minor_locator(
        NullLocator()
    )

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=False,
        top=False,
    )

    x_min = min(
        0.75,
        float(np.nanmin(lower)) * 0.94,
    )

    x_max = max(
        4.10,
        float(np.nanmax(upper)) * 1.04,
    )

    ax.set_xlim(
        x_min,
        x_max,
    )

    ax.set_yticks(y)

    ax.set_yticklabels(
        data["Display_Label"],
        fontsize=10.8,
    )

    ax.set_ylim(
        len(data) - 0.55,
        -0.55,
    )

    ax.tick_params(
        axis="y",
        length=0,
        pad=9,
    )

    ax.set_xlabel(
        "Adjusted hazard ratio (95% CI)\n"
        "Reference: low PRS / low NMR score",
        fontsize=11.5,
        labelpad=11,
    )

    ax.set_title(
        "Adjusted hazard ratios for incident CAD",
        loc="left",
        fontsize=15,
        fontweight="bold",
        pad=12,
    )

    ax.grid(
        axis="x",
        which="major",
        color=CNS["light_gray"],
        linewidth=0.7,
        alpha=0.75,
    )

    ax.grid(
        axis="y",
        visible=False,
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.subplots_adjust(
        left=0.39,
        right=0.97,
        top=0.91,
        bottom=0.20,
    )

    save_show(
        fig,
        "figure6C_revised_joint_adjusted_HR_CAD",
    )

except FileNotFoundError as error:
    print(
        "SKIP — missing figure source file: ",
        error,
    )


In [ ]:
try:

    source = find_unique('result5_final_joint_prs_nmr_observed_predicted_risk.csv')
    data = pd.read_csv(source); require_columns(data, ['Horizon_Years','Group_Clean','Group_Label','Observed_KM_Risk','Observed_KM_Risk_Lower95','Observed_KM_Risk_Upper95','Mean_Predicted_Risk'], source)
    data = data.loc[np.isclose(pd.to_numeric(data['Horizon_Years']),10.0)].copy(); data['Group_Clean']=pd.Categorical(data['Group_Clean'],GROUP_ORDER,ordered=True); data=data.sort_values('Group_Clean')
    y=np.arange(len(data)); obs=data['Observed_KM_Risk'].to_numpy(float)*100; lo=data['Observed_KM_Risk_Lower95'].to_numpy(float)*100; hi=data['Observed_KM_Risk_Upper95'].to_numpy(float)*100; pred=data['Mean_Predicted_Risk'].to_numpy(float)*100
    fig, ax=plt.subplots(figsize=(9.2,6.0)); ax.hlines(y,np.minimum(obs,pred),np.maximum(obs,pred),color=CNS['light_gray'],lw=1.3)
    ax.errorbar(obs,y,xerr=np.vstack([np.maximum(0,obs-lo),np.maximum(0,hi-obs)]),fmt='o',color=CNS['red'],ecolor=CNS['red'],capsize=2.5,markersize=6,markeredgecolor='white',label='Observed KM risk (95% CI)')
    ax.scatter(pred,y,s=48,marker='s',color=CNS['blue'],edgecolor='white',lw=0.6,label='Predicted')
    ax.set_yticks(y); ax.set_yticklabels(data['Group_Label']); ax.invert_yaxis(); ax.set_xlabel('10-year CAD risk (%)')
    ax.set_title('Observed and predicted 10-year CAD risk',loc='left',fontweight='bold'); clean_axis(ax,'x'); ax.legend(frameon=False,loc='upper right')
    data.to_csv(TABLE_DIR / 'figure6D_revised_observed_predicted_CAD.csv', index=False)
    fig.tight_layout(); save_show(fig, 'figure6D_revised_observed_predicted_10y_CAD_risk')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    source=find_unique("supplement_result5_aalen_johansen_joint_groups.csv"); data=pd.read_csv(source)
    require_columns(data,["Group_Clean","Time","Cumulative_Incidence"],source)
    fig,ax=plt.subplots(figsize=(12.5,7.2))
    for group in GROUP_ORDER:
        part=data.loc[data["Group_Clean"].astype(str).eq(group)].sort_values("Time")
        if not part.empty: ax.step(part["Time"],part["Cumulative_Incidence"].astype(float)*100,where="post",color=KM_COLORS[group],lw=1.7,label=GROUP_LABELS[group])
    ax.set_xlim(0,10); ax.set_xlabel("Follow-up time (years)"); ax.set_ylabel("CAD cumulative incidence (%)")
    ax.set_title("Competing-risk sensitivity: Aalen–Johansen",loc="left",fontweight="bold"); clean_axis(ax,"both")
    ax.legend(frameon=False,fontsize=10,bbox_to_anchor=(1.02,1),loc="upper left"); fig.tight_layout(); save_show(fig,"supplement_result5_aalen_johansen_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    data=result6_frame("result6_metabolite_main_effects_standardized_cns_final.csv")
    result6_volcano(data,"Metabolite main-effect associations","Log HR per 1-SD metabolite increment","figure7A_main_effect_volcano_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    data=result6_frame("result6_prs_metabolite_interactions_standardized_cns_final.csv")
    result6_volcano(data,"PRS–metabolite interaction signals","Log interaction HR: 1-SD PRS × 1-SD metabolite","figure7B_interaction_volcano_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    from matplotlib.ticker import FormatStrFormatter, MultipleLocator

    data = result6_frame(
        "result6_metabolite_main_effects_standardized_cns_final.csv"
    ).copy()

    required_columns = [
        "Metabolite_Label",
        "Beta",
        "HR",
        "HR_Lower95",
        "HR_Upper95",
        "FDR",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Figure 7Cmissing required columns: {missing_columns}"
        )

    for column in [
        "Beta",
        "HR",
        "HR_Lower95",
        "HR_Upper95",
        "FDR",
    ]:
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce",
        )

    data = data.dropna(
        subset=required_columns
    ).copy()

    invalid_ci = (
        (data["HR_Lower95"] > data["HR"])
        | (data["HR_Upper95"] < data["HR"])
        | (data["HR_Lower95"] <= 0)
    )

    if invalid_ci.any():
        raise RuntimeError(
            "Figure 7C contains an invalid HR or 95% CI."
        )

    pool = (
        data.loc[data["FDR"] < 0.05]
        .sort_values(
            ["FDR", "Metabolite_Label"]
        )
        .head(10)
        .copy()
    )

    if pool.empty:
        raise RuntimeError(
            "No main effects passed FDR < 0.05."
        )

    pool = (
        pool.sort_values(
            ["HR", "FDR"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    pool["Direction"] = np.where(
        pool["Beta"] >= 0,
        "Positive",
        "Negative",
    )

    pool.to_csv(
        TABLE_DIR
        / "figure7C_main_effect_forest_revised_source_data.csv",
        index=False,
    )

    figure_height = max(
        5.3,
        0.48 * len(pool) + 1.8,
    )

    fig = plt.figure(
        figsize=(10.0, figure_height)
    )

    grid = fig.add_gridspec(
        nrows=1,
        ncols=2,
        width_ratios=[3.5, 1.65],
        wspace=0.03,
    )

    ax = fig.add_subplot(grid[0, 0])

    value_ax = fig.add_subplot(
        grid[0, 1],
        sharey=ax,
    )

    y = np.arange(len(pool))

    for position in y:
        if position % 2 == 0:
            ax.axhspan(
                position - 0.5,
                position + 0.5,
                color="#F5F5F5",
                zorder=0,
            )

            value_ax.axhspan(
                position - 0.5,
                position + 0.5,
                color="#F5F5F5",
                zorder=0,
            )

    for position, row in pool.iterrows():
        point_color = (
            CNS["red"]
            if row["Beta"] >= 0
            else CNS["blue"]
        )

        ax.plot(
            [
                row["HR_Lower95"],
                row["HR_Upper95"],
            ],
            [position, position],
            color="#999999",
            linewidth=1.25,
            solid_capstyle="round",
            zorder=2,
        )

        ax.plot(
            [
                row["HR_Lower95"],
                row["HR_Lower95"],
            ],
            [
                position - 0.075,
                position + 0.075,
            ],
            color="#999999",
            linewidth=1.0,
            zorder=2,
        )

        ax.plot(
            [
                row["HR_Upper95"],
                row["HR_Upper95"],
            ],
            [
                position - 0.075,
                position + 0.075,
            ],
            color="#999999",
            linewidth=1.0,
            zorder=2,
        )

        ax.scatter(
            row["HR"],
            position,
            s=60,
            facecolor=point_color,
            edgecolor="white",
            linewidth=0.7,
            zorder=4,
        )

    ax.axvline(
        1.0,
        color="#444444",
        linestyle=(0, (2, 2)),
        linewidth=1.0,
        zorder=1,
    )

    ci_min = float(pool["HR_Lower95"].min())
    ci_max = float(pool["HR_Upper95"].max())
    ci_span = max(ci_max - ci_min, 0.01)

    x_lower = ci_min - 0.04 * ci_span
    x_upper = ci_max + 0.04 * ci_span

    x_lower = min(x_lower, 0.995)
    x_upper = max(x_upper, 1.005)

    ax.set_xscale("linear")
    ax.set_xlim(x_lower, x_upper)

    if x_upper - x_lower <= 0.20:
        tick_interval = 0.05
    else:
        tick_interval = 0.10

    ax.xaxis.set_major_locator(
        MultipleLocator(tick_interval)
    )

    ax.xaxis.set_major_formatter(
        FormatStrFormatter("%.2f")
    )

    ax.set_yticks(y)

    ax.set_yticklabels(
        pool["Metabolite_Label"],
        fontsize=10.5,
    )

    ax.set_ylim(
        len(pool) - 0.5,
        -0.5,
    )

    ax.set_xlabel(
        "Adjusted HR per 1-SD metabolite increment"
    )

    value_ax.set_xlim(0, 1)

    value_ax.set_ylim(
        len(pool) - 0.5,
        -0.5,
    )

    value_ax.text(
        0.03,
        1.025,
        "HR (95% CI)",
        transform=value_ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=9.8,
        fontweight="bold",
    )

    value_ax.text(
        0.98,
        1.025,
        "FDR",
        transform=value_ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=9.8,
        fontweight="bold",
    )

    def format_fdr(value):
        if value < 0.001:
            return f"{value:.1e}"
        return f"{value:.3f}"

    for position, row in pool.iterrows():
        hr_text = (
            f'{row["HR"]:.3f} '
            f'({row["HR_Lower95"]:.3f}–'
            f'{row["HR_Upper95"]:.3f})'
        )

        value_ax.text(
            0.03,
            position,
            hr_text,
            ha="left",
            va="center",
            fontsize=9.3,
            color=CNS["black"],
        )

        value_ax.text(
            0.98,
            position,
            format_fdr(row["FDR"]),
            ha="right",
            va="center",
            fontsize=9.1,
            color="#555555",
        )

    value_ax.tick_params(
        left=False,
        labelleft=False,
        bottom=False,
        labelbottom=False,
    )

    for spine in value_ax.spines.values():
        spine.set_visible(False)

    ax.set_title(
        "Top metabolite main-effect associations",
        loc="left",
        fontsize=15,
        fontweight="bold",
        pad=16,
    )

    ax.text(
        0,
        1.015,
        "Selected by lowest FDR and displayed in descending HR order",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=9.3,
        color="#666666",
    )

    ax.grid(
        axis="x",
        color="#E3E3E3",
        linewidth=0.65,
        alpha=0.75,
    )
    ax.grid(axis="y", visible=False)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.subplots_adjust(
        left=0.18,
        right=0.98,
        top=0.88,
        bottom=0.14,
    )

    save_show(
        fig,
        "figure7C_main_effect_forest_revised",
    )

except FileNotFoundError as error:
    print(
        "SKIP — missing figure source file: ",
        error,
    )


In [ ]:
# Figure 7D — Top PRS × metabolite interaction signals

try:
    data = result6_frame(
        "result6_prs_metabolite_interactions_standardized_cns_final.csv"
    )

    result6_interaction_forest(
        data=data,
        stem="figure7D_interaction_effects_revised",
        n=10,
    )

except FileNotFoundError as error:
    print(
        "SKIP — missing figure source file: ",
        error,
    )


In [ ]:
# Supplementary — ranked metabolite main effects
try:
    data=result6_frame('result6_metabolite_main_effects_standardized_cns_final.csv').sort_values(["FDR","Metabolite_Label"]).head(30).sort_values("FDR",ascending=False)
    y=np.arange(len(data)); fig,ax=plt.subplots(figsize=(9.2,max(6.2,0.28*len(data)+1.9)))
    ax.barh(y,data["MinusLog10FDR"],color=data["Color_Hex"],edgecolor="white",height=0.72); ax.axvline(-np.log10(0.05),color=CNS["gray"],ls="--",lw=1)
    ax.set_yticks(y); ax.set_yticklabels(data["Metabolite_Label"],fontsize=10); ax.set_xlabel(r"$-\log_{10}$(FDR)"); ax.set_title('Supplementary — ranked metabolite main effects',loc="left",fontweight="bold")
    clean_axis(ax,"x"); fig.tight_layout(); save_show(fig,'supplement_result6_ranked_main_effects_plot_only')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Supplementary — ranked PRS–metabolite interactions
try:
    data=result6_frame('result6_prs_metabolite_interactions_standardized_cns_final.csv').sort_values(["FDR","Metabolite_Label"]).head(30).sort_values("FDR",ascending=False)
    y=np.arange(len(data)); fig,ax=plt.subplots(figsize=(9.2,max(6.2,0.28*len(data)+1.9)))
    ax.barh(y,data["MinusLog10FDR"],color=data["Color_Hex"],edgecolor="white",height=0.72); ax.axvline(-np.log10(0.05),color=CNS["gray"],ls="--",lw=1)
    ax.set_yticks(y); ax.set_yticklabels(data["Metabolite_Label"],fontsize=10); ax.set_xlabel(r"$-\log_{10}$(FDR)"); ax.set_title('Supplementary — ranked PRS–metabolite interactions',loc="left",fontweight="bold")
    clean_axis(ax,"x"); fig.tight_layout(); save_show(fig,'supplement_result6_ranked_interactions_plot_only')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:

    source = find_unique('result6_prs_metabolite_interactions_discovery_replication.csv')
    data = pd.read_csv(source); required=['Metabolite_Label','coef','FDR','HR','HR_CI_Lower','HR_CI_Upper','Replication_Beta','Replication_P','Replication_HR','Replication_HR_CI_Lower','Replication_HR_CI_Upper']
    require_columns(data, required, source)
    for col in required[1:]: data[col]=pd.to_numeric(data[col],errors='coerce')
    sig=data.loc[data['FDR']<0.05].dropna(subset=['coef','HR','HR_CI_Lower','HR_CI_Upper']).copy()
    positive=sig.loc[sig['coef']>=0].nlargest(5,'coef'); negative=sig.loc[sig['coef']<0].nsmallest(5,'coef')
    selected=pd.concat([positive,negative],ignore_index=True); selected['Abs_Beta']=selected['coef'].abs(); selected=selected.sort_values(['coef'],ascending=False).reset_index(drop=True)
    if selected.empty: raise RuntimeError('No discovery interactions passed FDR < 0.05.')
    selected['Directional_Nominal_Replication']=(selected['Replication_P']<0.05)&(np.sign(selected['coef'])==np.sign(selected['Replication_Beta']))
    selected.to_csv(TABLE_DIR / 'supplementary_result6_interactions_discovery_validation_overlay.csv',index=False)
    y=np.arange(len(selected)); fig,ax=plt.subplots(figsize=(9.4,max(6.2,0.53*len(selected)+2.0)))
    for i,row in selected.iterrows():
        ax.errorbar(float(row['HR']),i-0.12,xerr=[[max(0,float(row['HR'])-float(row['HR_CI_Lower']))],[max(0,float(row['HR_CI_Upper'])-float(row['HR']))]],fmt='o',color=CNS['red'],ecolor=CNS['red'],capsize=3,markersize=6,label='Training discovery' if i==0 else None)
        ax.errorbar(float(row['Replication_HR']),i+0.12,xerr=[[max(0,float(row['Replication_HR'])-float(row['Replication_HR_CI_Lower']))],[max(0,float(row['Replication_HR_CI_Upper'])-float(row['Replication_HR']))]],fmt='s',color=CNS['blue'],ecolor=CNS['blue'],capsize=3,markersize=5.5,label='Validation replication' if i==0 else None)
    ax.axvline(1,color=CNS['gray'],lw=1); ax.set_xscale('log'); ax.set_yticks(y); ax.set_yticklabels(selected['Metabolite_Label'].map(clean_label)); ax.invert_yaxis()
    ax.set_xlabel('HR for PRS × metabolite interaction per 1-SD × 1-SD (95% CI)')
    ax.set_title('PRS–metabolite interaction signals: discovery and internal replication',loc='left',fontweight='bold')
    clean_axis(ax,'x'); ax.legend(frameon=False,loc='best'); fig.tight_layout(); save_show(fig,'supplementary_result6_interactions_discovery_validation_overlay')
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    source=find_unique("continuous_prs_x_nmr_score_test.csv"); data=pd.read_csv(source)
    require_columns(data,["Term","HR","HR_CI_Lower","HR_CI_Upper","P_Value"],source); row=data.iloc[0]
    fig,ax=plt.subplots(figsize=(7.2,3.2)); value=float(row["HR"]); lo=float(row["HR_CI_Lower"]); hi=float(row["HR_CI_Upper"])
    ax.errorbar(value,0,xerr=[[max(0,value-lo)],[max(0,hi-value)]],fmt="o",color=CNS["purple"],ecolor=CNS["purple"],capsize=4,markersize=7)
    ax.axvline(1,color=CNS["gray"],ls="--",lw=1); ax.set_xscale("log"); ax.set_yticks([0]); ax.set_yticklabels(["PRS × NMR_score"])
    ax.set_xlabel("Multiplicative interaction HR per 1-SD × 1-SD (95% CI)"); ax.set_title(f"Continuous interaction; P = {float(row['P_Value']):.3g}",loc="left",fontweight="bold")
    clean_axis(ax,"x"); fig.tight_layout(); save_show(fig,"supplement_continuous_prs_x_nmr_score_test_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    source=find_sensitivity_unique("figure2A_delta_uno_cindex.csv"); data=pd.read_csv(source)
    require_columns(data,["Comparison","Horizon_Years","Delta_Uno_C","Delta_Uno_C_Lower95","Delta_Uno_C_Upper95"],source)
    order=ordered_values(data,"Comparison",["Clinical + PRS vs Clinical","Clinical + NMR_score vs Clinical","Clinical + PRS + NMR_score vs Clinical","Integrated vs Clinical + PRS","Integrated vs Clinical + NMR_score"]); y=np.arange(len(order)); fig,ax=plt.subplots(figsize=(9.5,5.8))
    for horizon,color,marker,offset in [(5,CNS["teal"],"o",-0.11),(10,CNS["blue"],"s",0.11)]:
        part=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),horizon)].copy(); part["Order"]=part["Comparison"].map({x:i for i,x in enumerate(order)}); part=part.sort_values("Order"); value=part["Delta_Uno_C"].to_numpy(float); lo=part["Delta_Uno_C_Lower95"].to_numpy(float); hi=part["Delta_Uno_C_Upper95"].to_numpy(float)
        ax.errorbar(value,y+offset,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt=marker,color=color,ecolor=color,capsize=3,label=f"{horizon} years")
    ax.axvline(0,color=CNS["gray"]); ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlabel("Delta Uno IPCW C-index (95% CI)"); ax.set_title("Coverage90 sensitivity",loc="left",fontweight="bold"); clean_axis(ax,"x"); ax.legend(frameon=False)
    fig.tight_layout(); save_show(fig,"coverage90_delta_uno_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
try:
    source=find_sensitivity_unique("figure2C_delta_ipcw_ibs.csv"); data=pd.read_csv(source)
    require_columns(data,["Comparison","Horizon_Years","Delta_IBS","Delta_IBS_Lower95","Delta_IBS_Upper95"],source)
    order=ordered_values(data,"Comparison",["Clinical + PRS vs Clinical","Clinical + NMR_score vs Clinical","Clinical + PRS + NMR_score vs Clinical","Integrated vs Clinical + PRS","Integrated vs Clinical + NMR_score"]); y=np.arange(len(order)); fig,ax=plt.subplots(figsize=(9.5,5.8))
    for horizon,color,marker,offset in [(5,CNS["teal"],"o",-0.11),(10,CNS["blue"],"s",0.11)]:
        part=data.loc[np.isclose(pd.to_numeric(data["Horizon_Years"]),horizon)].copy(); part["Order"]=part["Comparison"].map({x:i for i,x in enumerate(order)}); part=part.sort_values("Order"); value=part["Delta_IBS"].to_numpy(float); lo=part["Delta_IBS_Lower95"].to_numpy(float); hi=part["Delta_IBS_Upper95"].to_numpy(float)
        ax.errorbar(value,y+offset,xerr=[np.maximum(0,value-lo),np.maximum(0,hi-value)],fmt=marker,color=color,ecolor=color,capsize=3,label=f"{horizon} years")
    ax.axvline(0,color=CNS["gray"]); ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlabel("Delta IPCW IBS; lower is better"); ax.set_title("Coverage90 sensitivity",loc="left",fontweight="bold"); clean_axis(ax,"x"); ax.legend(frameon=False)
    fig.tight_layout(); save_show(fig,"coverage90_delta_ibs_plot_only")
except FileNotFoundError as error:
    print('SKIP — missing figure source file: ', error)


In [ ]:
# Unified Figure S4 and Figure S9
# Plot only: reads saved DCA source files; no model refitting

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# 1. Input and output paths

DCA_SOURCE = Path("04_results/result5_final_cox_calibration_dca_audit/source_data/result5_final_standard_survival_dca.csv")
PRIMARY_SOURCE = RUN_ROOT / DCA_SOURCE
COVERAGE90_SOURCE = SENSITIVITY_ROOT / DCA_SOURCE
SOURCE_DATA_DIR = OUTPUT_ROOT / "source_data"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# 2. Unified journal-style colours

BLUE = "#3C5488"
RED = "#E64B35"
GRAY = "#9A9A9A"
DARK_GRAY = "#5A5A5A"
LIGHT_GRAY = "#E5E5E5"
BLACK = "#222222"

# 3. Unified plotting function

def plot_unified_dca(
    source_path,
    cohort_label,
    output_stem,
):
    source_path = Path(source_path)

    if not source_path.exists():
        raise FileNotFoundError(
            f"DCA source file not found:\n{source_path}"
        )

    data = pd.read_csv(source_path)

    required_columns = {
        "Strategy",
        "Threshold",
        "Net_Benefit",
        "Horizon_Years",
    }

    missing_columns = required_columns.difference(data.columns)

    if missing_columns:
        raise RuntimeError(
            "DCA source file is missing columns: "
            + ", ".join(sorted(missing_columns))
        )

    for column in ["Threshold", "Net_Benefit", "Horizon_Years"]:
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce",
        )

    # Ten-year results and prespecified 3%–15% threshold range.
    plot_data = data.loc[
        np.isclose(data["Horizon_Years"], 10.0)
        & data["Threshold"].between(
            0.03 - 1e-10,
            0.15 + 1e-10,
        )
    ].copy()

    plot_data["Threshold"] = (
        plot_data["Threshold"].round(6)
    )

    required_strategies = [
        "Clinical + PRS",
        "Clinical + PRS + NMR_score",
        "Treat all",
        "Treat none",
    ]

    missing_strategies = [
        strategy
        for strategy in required_strategies
        if strategy not in set(plot_data["Strategy"])
    ]

    if missing_strategies:
        raise RuntimeError(
            "Missing DCA strategies: "
            + ", ".join(missing_strategies)
        )

    # One row per threshold and strategy.
    wide = (
        plot_data.loc[
            plot_data["Strategy"].isin(required_strategies),
            ["Threshold", "Strategy", "Net_Benefit"],
        ]
        .pivot_table(
            index="Threshold",
            columns="Strategy",
            values="Net_Benefit",
            aggfunc="first",
        )
        .sort_index()
    )

    if len(wide) != 25:
        raise RuntimeError(
            "Expected 25 thresholds from 3% to 15% "
            f"in 0.5-point increments; found {len(wide)}."
        )

    if wide[required_strategies].isna().any().any():
        raise RuntimeError(
            "Missing net-benefit values within the "
            "prespecified threshold range."
        )

    threshold_percent = (
        wide.index.to_numpy(dtype=float) * 100
    )

    # Standard DCA: net benefit per 100 individuals.
    clinical_prs_nb_100 = (
        wide["Clinical + PRS"].to_numpy(dtype=float) * 100
    )

    joint_nb_100 = (
        wide["Clinical + PRS + NMR_score"]
        .to_numpy(dtype=float)
        * 100
    )

    treat_all_nb_100 = (
        wide["Treat all"].to_numpy(dtype=float) * 100
    )

    treat_none_nb_100 = (
        wide["Treat none"].to_numpy(dtype=float) * 100
    )

    # Incremental net benefit per 1,000 individuals.
    delta_nb_1000 = (
        wide["Clinical + PRS + NMR_score"].to_numpy(dtype=float)
        - wide["Clinical + PRS"].to_numpy(dtype=float)
    ) * 1000

    positive_thresholds = int(
        np.sum(delta_nb_1000 > 0)
    )

    # Save the plotted source data.

    source_output = wide.reset_index().copy()

    source_output.insert(
        1,
        "Threshold_Percent",
        source_output["Threshold"] * 100,
    )

    source_output[
        "Incremental_NB_Joint_minus_Clinical_PRS_per_1000"
    ] = delta_nb_1000

    source_output[
        "Positive_Incremental_NB"
    ] = delta_nb_1000 > 0

    source_csv = (
        SOURCE_DATA_DIR
        / f"{output_stem}_source_data.csv"
    )

    source_output.to_csv(
        source_csv,
        index=False,
    )

    # Draw the two-panel figure

    with plt.rc_context({
        "font.family": "DejaVu Sans",
        "font.size": 10.5,
        "axes.titlesize": 14,
        "axes.labelsize": 11.5,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 9.5,
        "axes.linewidth": 0.9,
        "savefig.dpi": 600,
    }):

        fig, (ax_top, ax_bottom) = plt.subplots(
            2,
            1,
            figsize=(8.6, 7.7),
            sharex=True,
            gridspec_kw={
                "height_ratios": [1.55, 1.0],
                "hspace": 0.12,
            },
        )

        # Overall title, identical for S4 and S9 except cohort label.
        fig.suptitle(
            f"Clinical utility of adding NMR score — {cohort_label}",
            x=0.11,
            y=0.985,
            ha="left",
            va="top",
            fontsize=15,
            fontweight="bold",
            color=BLACK,
        )

        # Panel A: standard DCA

        ax_top.plot(
            threshold_percent,
            treat_all_nb_100,
            color=GRAY,
            linestyle=(0, (5, 4)),
            linewidth=1.5,
            label="Treat all",
            zorder=1,
        )

        ax_top.plot(
            threshold_percent,
            treat_none_nb_100,
            color=DARK_GRAY,
            linestyle=(0, (1.5, 2.5)),
            linewidth=1.3,
            label="Treat none",
            zorder=1,
        )

        ax_top.plot(
            threshold_percent,
            clinical_prs_nb_100,
            color=BLUE,
            linestyle=(0, (6, 3)),
            linewidth=2.2,
            marker="o",
            markersize=4.6,
            markerfacecolor="white",
            markeredgecolor=BLUE,
            markeredgewidth=1.1,
            label="Clinical + PRS",
            zorder=3,
        )

        ax_top.plot(
            threshold_percent,
            joint_nb_100,
            color=RED,
            linestyle="-",
            linewidth=2.4,
            marker="D",
            markersize=4.5,
            markerfacecolor="white",
            markeredgecolor=RED,
            markeredgewidth=1.1,
            label="Clinical + PRS + NMR score",
            zorder=4,
        )

        # Fixed axis limits make S4 and S9 directly comparable.
        ax_top.set_ylim(-0.05, 2.85)

        ax_top.set_ylabel(
            "Net benefit\n(per 100 individuals)"
        )

        ax_top.text(
            -0.085,
            1.02,
            "A",
            transform=ax_top.transAxes,
            fontsize=14,
            fontweight="bold",
            ha="left",
            va="bottom",
        )

        ax_top.legend(
            frameon=False,
            ncol=2,
            loc="upper right",
            columnspacing=1.4,
            handlelength=3.2,
            borderaxespad=0.3,
        )

        # Panel B: incremental net benefit

        ax_bottom.axhline(
            0,
            color=DARK_GRAY,
            linestyle=(0, (1.5, 2.5)),
            linewidth=1.2,
            zorder=1,
        )

        ax_bottom.fill_between(
            threshold_percent,
            0,
            delta_nb_1000,
            where=delta_nb_1000 >= 0,
            interpolate=True,
            color=RED,
            alpha=0.13,
            zorder=1,
        )

        ax_bottom.fill_between(
            threshold_percent,
            0,
            delta_nb_1000,
            where=delta_nb_1000 < 0,
            interpolate=True,
            color=GRAY,
            alpha=0.13,
            zorder=1,
        )

        ax_bottom.plot(
            threshold_percent,
            delta_nb_1000,
            color=RED,
            linewidth=2.3,
            marker="o",
            markersize=4.4,
            markerfacecolor="white",
            markeredgecolor=RED,
            markeredgewidth=1.0,
            zorder=3,
        )

        # Same limits in both S4 and S9.
        ax_bottom.set_ylim(-0.50, 1.15)

        ax_bottom.set_ylabel(
            "Incremental net benefit\n(per 1,000 individuals)"
        )

        ax_bottom.set_xlabel(
            "10-year CAD risk threshold (%)"
        )

        ax_bottom.text(
            -0.085,
            1.02,
            "B",
            transform=ax_bottom.transAxes,
            fontsize=14,
            fontweight="bold",
            ha="left",
            va="bottom",
        )

        ax_bottom.text(
            0.98,
            0.90,
            "Joint model − Clinical + PRS",
            transform=ax_bottom.transAxes,
            ha="right",
            va="top",
            color=RED,
            fontsize=10.5,
            fontweight="bold",
        )

        ax_bottom.text(
            0.02,
            0.90,
            (
                f"Positive at {positive_thresholds}/"
                f"{len(delta_nb_1000)} thresholds"
            ),
            transform=ax_bottom.transAxes,
            ha="left",
            va="top",
            color=DARK_GRAY,
            fontsize=9.5,
        )

        # Shared axis formatting

        for axis in [ax_top, ax_bottom]:
            axis.set_xlim(3, 15)
            axis.set_xticks(
                np.arange(3, 15.1, 2)
            )

            axis.grid(
                axis="y",
                color=LIGHT_GRAY,
                linewidth=0.8,
                alpha=0.9,
            )

            axis.spines["top"].set_visible(False)
            axis.spines["right"].set_visible(False)
            axis.spines["left"].set_color(BLACK)
            axis.spines["bottom"].set_color(BLACK)

        ax_top.yaxis.set_major_locator(
            MultipleLocator(0.5)
        )

        ax_bottom.yaxis.set_major_locator(
            MultipleLocator(0.2)
        )

        # Note describing display clipping only.
        fig.text(
            0.5,
            0.018,
            (
                "The upper panel is scaled around the model curves; "
                "treat-all values outside the displayed range are "
                "clipped for display only."
            ),
            ha="center",
            va="bottom",
            fontsize=9,
            color="#666666",
        )

        fig.tight_layout(
            rect=[0.03, 0.055, 0.99, 0.955]
        )

        # Save PNG, SVG and PDF.
        for extension in ["png", "svg", "pdf"]:
            output_path = (
                FIGURE_DIR
                / f"{output_stem}.{extension}"
            )

            fig.savefig(
                output_path,
                dpi=600,
                bbox_inches="tight",
                facecolor="white",
            )

        plt.show()
        plt.close(fig)

    print(f"Saved figure: {FIGURE_DIR / output_stem}")
    print(f"Saved source data: {source_csv}")
    print(
        f"{cohort_label}: positive incremental net benefit at "
        f"{positive_thresholds}/{len(delta_nb_1000)} thresholds."
    )

    return source_output

# 4. Figure S4 — Coverage90 sensitivity cohort

figure_s4_data = plot_unified_dca(
    source_path=COVERAGE90_SOURCE,
    cohort_label="Coverage90 sensitivity cohort",
    output_stem="figureS4_coverage90_10y_DCA_unified",
)

# 5. Figure S9 — Primary cohort

figure_s9_data = plot_unified_dca(
    source_path=PRIMARY_SOURCE,
    cohort_label="Primary cohort",
    output_stem="figureS9_primary_10y_DCA_unified",
)
